# 教授名单批量补全（OpenAlex）

**用法：从上往下，每个格子点一次左边的 ▶ 播放键即可。不需要改任何代码。**

整份名单约 15–25 分钟跑完。中途断了重跑不会重复请求（有缓存）。

> ⚠️ **重要：每次重新打开或重新上传本 notebook，Colab 都会给一个全新的运行时，
> 之前上传的文件和生成的中间文件全部清空。**
>
> 所以务必**从第 1 格开始，从上往下依次运行**，
> 或直接用菜单 **代码执行程序 → 全部运行**（只会在第 2 步停下来让你选文件）。
>
> 如果中途报 `FileNotFoundError`，基本都是因为跳过了前面的步骤。用下面的「环境自检」格可以确认。

---
## 本版本更新说明（v2）

针对反馈的问题：

1. **「当前机构」显示为空** —— 之前用的是 OpenAlex 原始 `last_known_institutions` 字段（这个字段经常是空的），现在改用有兜底的 `effective_institutions`。
2. **机构别名转换正确但仍判定"模糊匹配"** —— 之前只要目标机构在作者**一生任何时候**出现过就算通过，导致多个不同的人都"沾边"。现在改成：优先要求目标机构是该作者**最近一次**的隶属（或在 OpenAlex 的 `last_known_institutions` 里），历史上出现过但不是最近一次的，单独标记为 `possible_move`，一律转人工核实，不会被当成 ok 自动采信。
3. **合并逻辑收紧**（两轮修正）：
   - 第一轮：合并前要求候选记录独立满足"当前机构"，不再仅凭主题重叠合并跨机构记录。
   - 第二轮（本次）：现实中大部分"同一人被拆成两条 OpenAlex 记录"是因为任职信息更新后系统重建了新 entity。因此合并条件改为——**先看其中一条记录"最后一次任职"是否出现在另一条记录的任职历史里**（判断是否为同一条时间线的前后接续），**如果是，再要求固定字段（姓名全称、ORCID 如果都有的话）完全一致**才允许合并；只要有一对候选不满足，整组都不合并、转人工确认。主题重叠检查仍保留作为附加的安全网，但不再单独作为合并依据。
4. **Prof/Assoc 等职称前缀** —— 确认本来就已经归一化处理，不是前面问题的成因（顺手删掉了一段没被用到的死代码）。

另外顺带修了两个后面步骤会遇到的坑：`confidence.py` 打分脚本原来引用了两个从未定义的变量，跑到「可信度评估」那一步必崩溃；`merge_to_excel.py` 里可信度那几列的字段名和 `confidence.py` 实际输出对不上，导致这几列永远是空的。这两个也已修复。

新增一列 **`OA_机构时效性`**（current / historical / unknown），以及一个新状态 **`possible_move`**（目标机构仅为历史隶属，需人工确认是否已转校）。合并记录的 `merge_note` 字段现在会写明是依据哪条证据链合并的，方便回溯核查。


---
## v3 更新：单个 OpenAlex ID 内部身份污染检测

上一版的机构时效性检查、跨entity合并检查，都是"拿候选人之间互相比较"，如果从始至终只搜到一个候选人（比如 Luo Jiang 这个 case），根本没有"第二个候选"可比对，之前所有检查都会被绕过。

新增 `_contamination_risk()`：不比较候选人之间，而是看**这一个** OpenAlex ID 自己的论文主题（`topics`/`x_concepts` 的 domain 分类）是否横跨了明显不相关的学科（例如医学 + 计算机同时出现），命中就在 `match_method` 后面追加 `_profile_contamination_risk`，`batch_enrich.py` 会把这类记录强制标记为 `needs_review_contaminated`，即使它是唯一候选、机构时效性也验证通过，也不会被当成 `ok` 自动采信。


---
## v4 更新

1. **`--field` / `--since` 不再硬编码**：配置区（第3步之后）新增下拉框选领域（`FIELD`，Colab 表单里可直接下拉选），`SINCE` 年份也做成了可改的表单输入，运行匹配的两处命令改为读取这两个变量，不再是写死的 `"finance"` / `"2022"`。
2. **新增"单机构 vs 多机构混杂"启发式**：候选池里如果恰好一个候选人只对应 1 个机构、其余候选人都对应 3 个以上机构，且这个"干净"候选人自己的论文主题没有跨越明显不相关的学科（复用了身份污染检测的逻辑判断"是否集中"），直接判定为目标人选（`matched_on_current_institution_clean_profile_heuristic`），不再需要 `field_score` 关键词命中——实测中候选人的论文主题经常不会直接包含 `finance` 这类词表词汇，原来单纯靠关键词打分打不破平局。
3. **新增两个统计列**：`OA_发表论文总数(历史总计)`（OpenAlex 记录的作者历史发表总数）与 `OA_近年发表数`（原来叫"近三年"，其实取决于 `--since` 设的年份，容易误导，已改名）。
4. **新增「论文明细」工作表**：最终导出的 xlsx 里现在有两个 sheet——原来的名单 sheet，加一个新的「论文明细」sheet，每篇论文一行，包含：教授姓名、名单机构、论文标题、发表年份、期刊/会议、**该作者本人在这篇论文上的所属机构**（不是他现在的机构，是发这篇文章时的机构）、文章链接、DOI、合作者。

**关于被引数和近年文章的确认**：`OA_被引总数` 是 OpenAlex 给出的作者**历史总被引**，不是近年窗口内的被引数（目前没有单独统计"近年被引数"，如果需要可以再加）；`OA_近年发表数` 对应的年份范围由配置区的 `SINCE` 决定，不是固定"2022年之后"或"近3年"，改 `SINCE` 的值就能调整。


---
## v5 修复：机构别名缓存"学了但没记住"的 bug

之前反馈"机构一致性检查用的是简单子串匹配"，往下查发现根因有两层：

1. `merge_to_excel.py` 是完全独立的脚本，压根没 `import resolve_v2`，最后那个"机构是否一致"的核对是我另外重写的一套弱子串匹配，没有复用前面已经做好的别名表基础设施。**现已修复**：`merge_to_excel.py` 改为 `import resolve_v2 as rv`，直接调用 `resolve_institution()` 做核对，缩写会先转换成 OpenAlex canonical 全称再比较。

2. 更深一层：`resolve_v2.py` 里 `_save_alias()`（负责把新解析出来的机构写回 `institution_aliases.json`，讲道理这样以后同一个机构就不用重新联网搜了）里有一行判断逻辑写反了——`if key in d and not d[key].get("learned", True): return`。种子表里预置的 HKU / NTU / HKUST 这些条目本来就是 `"learned": false`（表示"还没配上 openalex_id"），但这行代码的效果恰恰是：**只要条目已经在文件里且标记是 `learned: false`，就直接放弃写入**——正好把最需要被补全的那批条目锁死了，导致这几个机构每次都得重新联网搜索，`openalex_id` 永远填不进去。已删除这条多余且写反的保护逻辑（这个函数本来就只会在"之前没有可用 id"时被调用，不存在会覆盖已验证结果的风险）。

用模拟数据验证过：修复后 `resolve_institution("HKU", ...)` 第一次成功解析后会把 `openalex_id` 真正写回磁盘上的 `institution_aliases.json`；`merge_to_excel.py` 作为全新子进程运行时（和 notebook 里 `batch_enrich.py` → `merge_to_excel.py` 两步分别 `subprocess.run` 的执行方式一致）能读到这个缓存，之前"HKU vs University of Hong Kong"被误判为机构不一致的问题不再出现。


---
## v6 修复：干净候选启发式被自己的一个残留条件误伤

上一版把"单机构 vs 多机构混杂"启发式的判断依据从"关键词命中 `field_score`"改成了"学科是否集中（复用污染检测）"之后，函数内部已经不再用到 `field` 这个参数——但调用它的地方忘了把 `if field and len(work_pool) > 1:` 这个已经过时的门槛去掉。结果是：只要调用时 `field` 是空字符串或 `None`，这条启发式会被整个跳过，哪怕候选人的机构数据已经足够清晰地判断出来了。已改为 `if len(work_pool) > 1:`，不再要求 `field` 非空。

用模拟数据复现过：`field=""` / `field=None` 时旧代码确实停在 `ambiguous_same_institution`（和反馈的现象一致），`field="finance"` 时旧代码反而是正常的；修复后三种情况全部正确解析到干净候选人。


---
## v7：全量代码梳理，清掉死代码，堵住几个"新状态没被旧逻辑感知到"的漏洞

按要求把 `.skill` 里所有脚本过了一遍，找已经被取代但没删的旧代码，以及新增状态后忘记同步更新的旧判断逻辑：

**删掉的死代码**（定义了但确认没人调用）：
- `resolve_v2.py`：`_inst_ok`、`_topics_of`（分别被更完善的 `_affil_ids`/`verify_institution`、`_topic_set` 取代）
- `confidence.py`：`_inst_tokens` 及其专用的 `ALIASES` 导入（已被 `_inst_agrees`+`same_institution` 取代）；`METHOD_BASE` 里三个 `resolve_v2.find_author` 现在永远不会返回的旧字符串
- `fetch_openalex.py`：独立维护的弱版 `find_author()`（不做机构ID解析/时效性判断/污染检测，功能被 `resolve_v2.find_author` 全面取代）——`main()` 改为直接调用 `resolve_v2.find_author`
- `batch_enrich.py` / `merge_to_excel.py`：`rejected_field_mismatch` 这个从未被实际产生过的状态分支

**修的"新状态没被旧逻辑感知到"**：
- `openalex_links.py` 生成"待人工确认.md"时默认只筛 `--status ambiguous` 前缀，新加的 `possible_move`/`needs_review_contaminated` 状态因为前缀不同，之前会被漏掉。现在默认改成"非 ok 就都算"，和 `merge_to_excel.py`/`batch_enrich.py` 结尾汇总的判定口径统一。
- `batch_enrich.py` 结尾打印"需要人工复核"名单时也只筛了 `ambiguous`，同样漏掉了新状态，一并修了。
- `confidence.py` 里"近三年论文加分"的判断硬编码成了 `>= 2023`，会随时间推移逐年过时，改成基于当前日期动态计算。

**消除重复维护、容易漂移的数据**：
- `resolve_identity.py`（独立的单人查询工具）原来调的是 `fetch_openalex.py` 那个已删除的弱版 `find_author`，会直接报错——已改为调用 `resolve_v2.find_author`，和批量流程用同一套解析逻辑，保留了它原有的 ORCID 交叉验证。
- `reverse_lookup.py` 原来自己维护一份 `FIELD_TOPICS` 领域关键词表，和 `resolve_v2.FIELD_HINTS` 已经有几处轻微不同步（比如 "om" 领域一个有 "operations research" 一个没有），现在改为直接复用 `resolve_v2.FIELD_HINTS`，避免以后改一处忘另一处。

全部改动在隔离的沙盒环境里重跑了此前建立的回归测试（干净候选启发式、拆分实体合并判定、身份污染检测、field 为空场景、机构别名一致性检查），结果与预期一致，未引入功能性回归。


---
## v8 修复：干净候选启发式用"全职业生涯机构数"当门槛，几乎打不到任何真人

上一版"单机构 vs 多机构混杂"启发式判断机构数用的是 `_affil_ids()`——这个函数统计的是候选人**一辈子出现过的所有机构**，不是候选列表里实际展示的那份"机构=[...]"（那份显示的其实是 OpenAlex 的 `last_known_institutions`，即当前/近期任职）。

问题是：一个真教授只要博士不是在现在这家机构读的，`_affil_ids()` 就会算出 2 个机构（母校 + 现在的学校），永远够不到 `<=1` 的"干净"门槛——这个 bug 对几乎所有有正常职业履历的人都成立，不是个别情况，所以启发式实际上几乎从来没真正生效过。

改成用 `_current_institution_names()`——和候选列表实际展示给你看的那份数据同一个来源（`last_known_institutions`，为空则退回 `affiliations` 列表）。同时注意到一个连带问题：`_fmt` 展示用的这份数据本来就把 `affiliations` 兜底截到前 3 条（纯粹为了打印好看），如果计数也复用这个截断，"混杂"候选人的机构数永远算不出超过 3，`noisy` 判定又会失效——所以计数时特意不做这个截断，只有展示时才截。

用一个更贴近真实情况的场景验证过：候选人博士期间在另一所学校、现在只在 HKU 任职（`last_known_institutions` 只有 HKU 一条，但完整履历里有 2 个机构）——修复前这种情况会被误判为"不干净"而跳过启发式，修复后能正确识别。同时重跑了之前建立的全部 6 项回归测试，结果都和预期一致。


---
## v9：邮箱做成可填变量 + 一轮运行时逻辑漏洞排查

**MAILTO 也做成表单变量**：和 SINCE/FIELD 一样用 Colab 的 `#@param` 语法，配置区可以直接改，不用去 6 处代码里找哪里写死了邮箱（顺手把 SHEET 也改成了文本输入框）。

**排查中发现并修复的运行时问题**：

1. **缓存 key 没带上 `--field`/`--since`** —— `batch_enrich.py` 判断"这个人是不是已经处理过"只看姓名+机构，换个领域或改个起始年份重新跑，会直接命中旧缓存、静默返回上一次的结果。现在 key 里加上了这两个参数；另外加了 `--no-cache` 选项，不用手动删缓存文件也能强制重新拉取。

2. **API 请求没有重试** —— 遇到限流（429）或偶发 5xx，之前是直接抛异常，而大多数调用方是 `try/except: continue`，相当于静默丢弃这次搜索，可能让本来能解析出来的人被误判成 `not_found`，且完全没有提示。现在 `resolve_v2.py` 和 `fetch_openalex.py` 的请求函数都加了指数退避重试。

3. **`merge_to_excel.py` 按姓名回填结果时没带机构** —— 如果名单里有两个同名不同机构的人（常见姓氏很容易撞上），后一个人的数据会静默覆盖前一个，导致两行都拿到同一份、其中一份是错的数据。现在改成按"姓名+机构"组合匹配，且用模拟的同名不同机构数据验证过不再互相覆盖。

4. **最后一步没传 `--mailto`** —— notebook 里生成最终 xlsx 那一步调用 `merge_to_excel.py` 时漏了 `--mailto` 参数，导致机构一致性检查兜底联网解析时没走"礼貌池"，容易被限流。已补上。

5. **`per_page` 应为 `per-page`** —— `fetch_openalex.py` 的 `works()` 用的是 Python 关键字参数 `per_page=`（下划线），但 OpenAlex API 实际认的参数名是 `per-page`（连字符）。因为关键字参数不能带连字符，这个参数其实从来没真正发出去过，OpenAlex 会静默忽略、退回默认的每页 25 条——**发表论文超过 25 篇的教授，论文明细和近年论文数量一直在被默默截断**。已改成和代码库其余位置一致的写法，用模拟请求验证了实际发出的参数确实带上了 `per-page: 200`。



---
## 第 1 步：安装依赖

点 ▶ 运行。大约 20 秒。

In [ ]:
!pip install -q requests openpyxl
print('依赖安装完成')

---
## 第 2 步：上传你的 Excel

点 ▶ 之后会出现「选择文件」按钮，选你的 `教授论文与聚焦度分析_vXX.xlsx`。

In [ ]:
from google.colab import files
up = files.upload()
XLSX = list(up.keys())[0]
print('已上传：', XLSX)

---
## 第 3 步：填写你的邮箱

OpenAlex 要求调用方提供联系邮箱（进入「礼貌池」，速度更快更稳定）。
**把下面的邮箱改成你自己的**，然后点 ▶。

In [ ]:
MAILTO = "your-email@example.com"  #@param {type:"string"}
# ↑ 你自己的邮箱：OpenAlex 建议带上邮箱调用它们的 API（所谓 "polite pool"），
#   请求会更稳定、限流更宽松。换成你自己的邮箱即可，不需要注册、不会收到邮件。

SHEET = "全部教授_论文与聚焦度"  #@param {type:"string"}
# ↑ 工作表名，要和你上传的 xlsx 里实际的 sheet 名一致

SINCE = 2022  #@param {type:"integer"}
# ↑ 只统计这一年起的论文（不是固定"近3年"，改这个数字就能调整窗口）

FIELD = "finance"  #@param ["finance", "accounting", "economics", "is", "om"]
# ↑ 领域：用来打破候选人歧义、判断某候选人的论文是否"集中在这个领域内"。
#   在 Colab 里这一格右侧会出现下拉框，选完重新运行本格即可。
#   对应 resolve_v2.py 的 FIELD_HINTS，选错不会报错，只是打分/启发式匹配会失效。

print(f'配置完成：MAILTO={MAILTO}，SHEET={SHEET}，SINCE={SINCE}，FIELD={FIELD}')


---
## 第 4 步：载入脚本

直接点 ▶，不用看内容。

In [ ]:
%%writefile fetch_openalex.py
#!/usr/bin/env python3
"""Fetch works, coauthors, affiliation timeline and yearly output from OpenAlex.

Affiliation year-ranges give the academic start year; counts_by_year reveals
output slowdowns that a static publication list hides.

Usage:
  python fetch_openalex.py --name "Jane Doe" --institution "Yale" --mailto you@x.com
  python fetch_openalex.py --author-id A5023888391 --mailto you@x.com
"""
import argparse, json, sys, time
import requests

BASE = "https://api.openalex.org"


def short_id(obj):
    """OpenAlex ids are URLs, and some records carry a null id. Never assume."""
    v = obj if isinstance(obj, str) else (obj or {}).get("id")
    return v.rsplit("/", 1)[-1] if isinstance(v, str) and v else None


def _get(path, mailto, **params):
    """GET with a short retry/backoff for rate limits and transient errors —
    see resolve_v2._get for why this matters even for a single failed call."""
    if mailto:
        params["mailto"] = mailto
    last_exc = None
    for attempt in range(4):
        try:
            r = requests.get(f"{BASE}/{path}", params=params, timeout=45)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(1.5 * (2 ** attempt))
                continue
            r.raise_for_status()
            time.sleep(0.15)
            return r.json()
        except requests.exceptions.RequestException as e:
            last_exc = e
            time.sleep(1.5 * (2 ** attempt))
    raise last_exc or RuntimeError(f"OpenAlex request failed after retries: {path}")


def profile(author_id, mailto):
    a = _get(f"authors/{author_id}", mailto)
    affil = []
    for x in a.get("affiliations") or []:
        affil.append({"institution": (x.get("institution") or {}).get("display_name"),
                      "years": sorted(x.get("years") or [])})
    counts = {c["year"]: {"works": c["works_count"], "cites": c["cited_by_count"]}
              for c in a.get("counts_by_year") or []}
    academic_start = min([min(x["years"]) for x in affil if x["years"]], default=None)
    all_insts = [x["institution"] for x in affil if x.get("institution")]
    last = [i.get("display_name") for i in (a.get("last_known_institutions") or [])]
    # last_known_institutions is frequently empty even when the affiliation history is
    # populated; fall back so downstream display never looks like "no institution".
    effective = last or all_insts[:3]
    return {
        "id": author_id,
        "name": a.get("display_name"),
        "orcid": a.get("orcid"),
        "works_count": a.get("works_count"),
        "cited_by_count": a.get("cited_by_count"),
        "last_known_institutions": last,
        "affiliation_institutions": all_insts,
        "effective_institutions": effective,
        "affiliations": affil,
        "earliest_affiliation_year": academic_start,
        "counts_by_year": counts,
        "topics": [t.get("display_name") for t in (a.get("topics") or [])[:10]],
    }


def works(author_id, mailto, since=None, limit=200):
    """Fetch up to `limit` works (single page — OpenAlex's per-page max is 200,
    so a professor with more than 200 papers within the `since` window would
    still be truncated; that's rare enough in practice not to warrant full
    cursor pagination here, unlike reverse_lookup.py's institution-wide scan).
    """
    f = f"author.id:{author_id}"
    if since:
        f += f",from_publication_date:{since}-01-01"
    # NOTE: OpenAlex's actual parameter name is "per-page" (hyphen), which
    # can't be written as a Python keyword argument — passing per_page=...
    # instead silently sends a parameter OpenAlex doesn't recognise, and it
    # falls back to its own default page size (25) with no error. Must go
    # through **{"per-page": ...} like every other _get call in this project.
    d = _get("works", mailto, **{"filter": f, "per-page": min(limit, 200),
                                 "sort": "publication_year:desc"})
    out = []
    for w in d.get("results", []):
        wid = short_id(w)
        # Find THIS author's own authorship entry to read the institution(s) they
        # were affiliated with on THIS specific paper — not their current/overall
        # institution, which can differ from a paper written years ago.
        my_insts, coauthors = [], []
        for au in w.get("authorships") or []:
            au_id = short_id(au.get("author"))
            au_name = (au.get("author") or {}).get("display_name")
            if au_id == author_id:
                my_insts = [i.get("display_name") for i in (au.get("institutions") or [])
                           if i.get("display_name")]
            else:
                coauthors.append({"id": au_id, "name": au_name})
        out.append({
            "id": wid,
            "url": f"https://openalex.org/{wid}" if wid else None,
            "title": w.get("display_name"),
            "year": w.get("publication_year"),
            "venue": ((w.get("primary_location") or {}).get("source") or {}).get("display_name"),
            "type": w.get("type"),
            "doi": w.get("doi"),
            "is_published": (w.get("type") == "article"
                             and ((w.get("primary_location") or {}).get("source") is not None)),
            "institutions": my_insts,          # this author's affiliation ON this paper
            "coauthors": coauthors,             # everyone else on the paper
        })
    return out


def main():
    import os, sys as _sys
    _sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import resolve_v2 as rv   # same resolver batch_enrich.py uses — institution-ID

    p = argparse.ArgumentParser()
    p.add_argument("--name"); p.add_argument("--institution")
    p.add_argument("--author-id"); p.add_argument("--mailto")
    p.add_argument("--field", help="e.g. finance — helps break ties among candidates")
    p.add_argument("--since", help="YYYY, restrict works")
    a = p.parse_args()
    if not a.author_id:
        if not a.name:
            p.error("need --name or --author-id")
        cands, how = rv.find_author(a.name, a.institution, a.mailto, field=a.field)
        if len(cands) != 1:
            print(json.dumps({"status": "ambiguous", "how": how, "candidates": cands[:10],
                              "note": "Resolve by institution/topic before proceeding. "
                                      "Do not guess."}, ensure_ascii=False, indent=2))
            return
        a.author_id = cands[0]["id"]
        print(f"# resolved to {a.author_id} ({how})", file=sys.stderr)
    prof = profile(a.author_id, a.mailto)
    prof["works"] = works(a.author_id, a.mailto, a.since)
    print(json.dumps(prof, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile resolve_v2.py
#!/usr/bin/env python3
"""Precise author resolution: institution ID filtering + field constraint + name variants.

String-matching an institution abbreviation ("NUS") against OpenAlex full names
("National University of Singapore") fails silently and makes every row ambiguous.
This module resolves the institution to an OpenAlex ID first, then filters on it.
"""
import json, os, re, time, requests

BASE = "https://api.openalex.org"

# Alias table lives on disk so that mappings learned on one run persist to the next,
# and so that a roster using unfamiliar abbreviations teaches the skill rather than
# requiring every user to edit source. Layered lookup:
#   1. local alias file  2. OpenAlex acronym/alternative-name fields  3. plain search
def _find_alias_path():
    """Locate the alias file across layouts (skill dir, flat dir, cwd)."""
    here = os.path.dirname(os.path.abspath(__file__))
    for p in (os.path.join(here, "..", "assets", "institution_aliases.json"),
              os.path.join(here, "assets", "institution_aliases.json"),
              os.path.join(os.getcwd(), "assets", "institution_aliases.json"),
              os.path.join(here, "institution_aliases.json")):
        if os.path.exists(p):
            return os.path.abspath(p)
    # none exist yet: prefer a writable location next to the script
    return os.path.abspath(os.path.join(here, "assets", "institution_aliases.json"))


_ALIAS_PATH = _find_alias_path()


def _load_aliases():
    try:
        d = json.load(open(_ALIAS_PATH, encoding="utf-8"))
        return {k: v for k, v in d.items() if not k.startswith("_")}
    except Exception:
        return {}


def _save_alias(key, oa_id, display):
    """Persist a newly learned abbreviation so later runs (and other users) benefit.

    resolve_institution() only ever reaches this function when it did NOT already
    have a usable openalex_id cached for `key` (a cached id short-circuits before
    getting here) — so there is nothing to protect from being overwritten. Seed
    entries in the alias file start as {"learned": false, no openalex_id}
    specifically so this call can complete them.
    """
    try:
        d = json.load(open(_ALIAS_PATH, encoding="utf-8"))
    except Exception:
        d = {}
    d[key] = {"openalex_id": oa_id, "display_name": display, "learned": True}
    try:
        os.makedirs(os.path.dirname(_ALIAS_PATH), exist_ok=True)
        json.dump(d, open(_ALIAS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    except Exception:
        pass


ALIASES = _load_aliases()
_inst_cache, _sess = {}, requests.Session()

# ---------------------------------------------------------------------------
# Institution identity
#
# Institution names are unique, so once an abbreviation has been expanded the
# comparison should be exact rather than similarity-based. Fuzzy overlap produces
# false positives on shared words: "Hong Kong University of Science and Technology"
# and "University of Science and Technology of China" share three of five tokens.
#
# Abbreviations are almost always the initials of the significant words, which gives
# an independent way to confirm that an expansion is the right one.
# ---------------------------------------------------------------------------
_INST_STOP = {"of", "the", "and", "at", "for", "in", "de", "la", "des", "du"}


def norm_inst(name):
    """Lowercase, drop punctuation and leading 'the', collapse whitespace."""
    n = re.sub(r"[^\w\s]", " ", str(name or "").lower())
    n = re.sub(r"\s+", " ", n).strip()
    return re.sub(r"^the ", "", n)


def initials(full_name):
    """First letters of the significant words: 'Hong Kong University of Science and
    Technology' -> 'hkust'."""
    words = [w for w in norm_inst(full_name).split() if w not in _INST_STOP]
    return "".join(w[0] for w in words if w)


def initials_match(abbrev, full_name):
    """Does an abbreviation look like the initials of this full name?"""
    a = re.sub(r"[^a-z]", "", str(abbrev or "").lower())
    if not a:
        return False
    ini = initials(full_name)
    if a == ini:
        return True
    # tolerate abbreviations that drop a trailing qualifier: HKU vs HKUST-style cases
    return len(a) >= 3 and ini.startswith(a)


def same_institution(roster_inst, record_inst):
    """Identity test after expansion.

    Exact match first, then containment (handles a leading 'The' and sub-units whose
    name embeds the parent). Finally the abbreviation itself, since OpenAlex sometimes
    stores departments as e.g. 'HKUST Business School'.
    """
    raw = str(roster_inst or "").strip()
    a = norm_inst((ALIASES.get(raw) or {}).get("display_name") or raw)
    b = norm_inst(record_inst)
    if not a or not b:
        return False
    if a == b or a in b or b in a:
        return True
    abbr = re.sub(r"[^a-z]", "", raw.lower())
    return len(abbr) >= 3 and abbr in b.replace(" ", "")




def _get(path, mailto=None, **params):
    """GET with a short retry/backoff for rate limits and transient errors.

    Callers mostly wrap _get() in try/except: continue (skip this search
    attempt rather than crash the whole batch) — which means a single
    unretried 429 or transient 5xx would silently drop a legitimate search
    attempt and could turn a resolvable person into a false "not_found" with
    no indication anything went wrong. Retrying here, once, centrally, is
    cheaper than reasoning about that at every call site.
    """
    if mailto:
        params["mailto"] = mailto
    last_exc = None
    for attempt in range(4):
        try:
            r = _sess.get(f"{BASE}/{path}", params=params, timeout=45)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(1.5 * (2 ** attempt))
                continue
            r.raise_for_status()
            time.sleep(0.12)
            return r.json()
        except requests.exceptions.RequestException as e:
            last_exc = e
            time.sleep(1.5 * (2 ** attempt))
    raise last_exc or RuntimeError(f"OpenAlex request failed after retries: {path}")


def resolve_institution(name, mailto=None):
    """Roster institution string -> (OpenAlex institution id, canonical name).

    Handles abbreviations generically: OpenAlex institution records carry
    `display_name_acronyms` and `display_name_alternatives`, so an unknown
    abbreviation can usually be resolved without a hand-maintained table.
    Successful resolutions are written back to the alias file.
    """
    if not name:
        return None, None
    key = str(name).strip()
    if key in _inst_cache:
        return _inst_cache[key]

    # layer 1: local alias file
    hit = ALIASES.get(key)
    query = key
    if hit:
        if hit.get("openalex_id"):
            out = (hit["openalex_id"], hit.get("display_name"))
            _inst_cache[key] = out
            return out
        query = hit.get("display_name") or key

    def _search(q):
        try:
            return (_get("institutions", mailto, **{"search": q, "per-page": 10}).get("results") or [])
        except Exception:
            return []

    res = _search(query)
    chosen = None
    if res:
        kl = key.lower()
        # layer 2a: the institution's own registered acronym
        for r in res:
            names = [str(x).lower() for x in (r.get("display_name_acronyms") or [])]
            names += [str(x).lower() for x in (r.get("display_name_alternatives") or [])]
            if kl in names:
                chosen = r
                break
        # layer 2b: the abbreviation matches the initials of the full name
        if chosen is None:
            for r in res:
                if initials_match(key, r.get("display_name")):
                    chosen = r
                    break
        # layer 2c: exact name equality
        if chosen is None:
            for r in res:
                if norm_inst(r.get("display_name")) == norm_inst(query):
                    chosen = r
                    break
        chosen = chosen or res[0]

    if not chosen:
        _inst_cache[key] = (None, None)
        return None, None
    oa_id = _short_id(chosen)
    if not oa_id:
        _inst_cache[key] = (None, None)
        return None, None
    out = (oa_id, chosen.get("display_name"))
    _inst_cache[key] = out
    _save_alias(key, oa_id, chosen.get("display_name"))
    return out


# Western given names commonly adopted by scholars of Chinese/Korean heritage.
# Rosters often record "Ka Chung Boris NG" while publications use "Ka Chung Ng",
# so the adopted name must be dropped to find the real publication record.
_ADOPTED = set("""alex alan amy andy angela annie anthony ben betty bill bob boris brian
bruce carol cathy charlie cherry chris cindy claire coco daisy dan daniel david eddie
eden elaine ellen emily eric eva fiona frank gary george grace helen henry ivy jack
jacky james jane janet jason jeff jenny jerry jessica jimmy joe john johnny joyce judy
julia julie karen kate kathy keith kelly ken kenny kevin kitty lambert larry laura
leo lily linda lionel lisa louis lucy luke mandy marco mark martin mary matt max may
michael michelle mike nancy nick nicole olivia oscar patrick paul peter philip rachel
ray raymond rebecca richard rick robert roger ronald rose roy ruby sam samuel sandy
sarah sean shirley simon simba sophia stanley stella steve steven sunny susan terry
thomas tiffany tim tina toby tom tony tracy vicky victor vincent vivian wendy william
willy winnie yvonne""".split())


def name_variants(raw):
    """Produce ordered name variants, most likely first.

    Handles three roster conventions at once:
      * titles and CJK characters mixed in           -> stripped
      * surname written in ALL CAPS                  -> used to fix word order
      * an adopted Western given name inserted       -> dropped, since publications
        typically carry only the romanised given name
    """
    n = re.sub(r"\b(Prof|Professor|Dr|Assoc|Asst|Associate|Assistant|Mr|Ms|Mrs)\.?\b", " ",
               raw, flags=re.I)
    n = re.sub(r"[（(].*?[)）]", " ", n)
    n = "".join(ch for ch in n if not ("\u4e00" <= ch <= "\u9fff"))
    n = re.sub(r"[^A-Za-z\-'\s]", " ", n)
    toks = [t for t in n.split() if len(t) > 1]
    if not toks:
        return []

    # a single ALL-CAPS token (2+ letters) is almost always the surname
    caps = [t for t in toks if t.isupper() and len(t) > 1]
    surname = caps[0] if len(caps) == 1 else None
    given = [t for t in toks if t is not surname] if surname else []

    out = []

    def add(*parts):
        v = " ".join(p for p in parts if p).strip()
        if v and v.lower() not in {x.lower() for x in out}:
            out.append(v)

    if surname:
        core = [g for g in given if g.lower() not in _ADOPTED]
        add(*core, surname)                       # Ka Chung Ng      <- usually correct
        add(surname, *core)                       # Ng Ka Chung
        if core != given:
            add(*given, surname)                  # Ka Chung Boris Ng
    else:
        add(*toks)
        if len(toks) >= 2:
            add(*toks[::-1])
            add(toks[-1], toks[0])
            core = [t for t in toks if t.lower() not in _ADOPTED]
            if core != toks and core:
                add(*core)
                add(*core[::-1])
    return out


FIELD_HINTS = {
    "finance": ["finance", "financial economics", "asset pricing", "corporate finance",
                "capital market", "investment", "banking"],
    "accounting": ["accounting", "auditing", "disclosure", "financial reporting"],
    "economics": ["economics", "econometrics", "macroeconomics"],
    "is": ["information systems", "information technology", "e-commerce", "platform"],
    "om": ["operations management", "supply chain", "logistics"],
}


def _tokens(n):
    n = re.sub(r"[^A-Za-z\-\s]", " ", str(n or "")).replace("-", " ")
    return [t.lower() for t in n.split() if len(t) > 1]


def name_compatible(query, candidate):
    """Reject fuzzy matches that share only a surname.

    OpenAlex `display_name.search` is token-based and fuzzy: querying
    "Ka Chung Ng" also returns "Ka Wai Ng", "Ka Po Ng", "Ka Lok Ng" and so on.
    Those are different people. Require that no given name conflicts: every given
    token of the shorter name must appear in the longer one (an initial counts as
    a match for a full token beginning with it).
    """
    q, c = _tokens(query), _tokens(candidate)
    if not q or not c:
        return False
    if q[-1] != c[-1]:                      # surnames must agree
        return False
    qg, cg = set(q[:-1]), set(c[:-1])
    if not qg or not cg:
        return bool(qg or cg) is False or qg == cg
    short, long_ = (qg, cg) if len(qg) <= len(cg) else (cg, qg)
    for t in short:
        if t in long_:
            continue
        if len(t) == 1 and any(x.startswith(t) for x in long_):
            continue
        if any(len(x) == 1 and t.startswith(x) for x in long_):
            continue
        return False                        # a given name conflicts -> different person
    return True


def _split_name(raw):
    """-> (given_tokens, surname). Uses an ALL-CAPS token as the surname when present."""
    n = re.sub(r"\b(Prof|Professor|Dr|Assoc|Asst|Associate|Assistant|Mr|Ms|Mrs)\.?\b", " ",
               raw or "", flags=re.I)
    n = re.sub(r"[（(].*?[)）]", " ", n)
    n = "".join(ch for ch in n if not ("\u4e00" <= ch <= "\u9fff"))
    n = n.replace("-", " ").replace("'", " ")
    toks = [t.strip() for t in re.split(r"[\s,]+", n) if t.strip() and len(t) > 1]
    if not toks:
        return [], ""
    caps = [t for t in toks if t.isupper() and len(t) > 1]
    if len(caps) == 1:
        sur = caps[0]
        given = [t for t in toks if t is not sur]
    else:
        sur = toks[-1]
        given = toks[:-1]
    return [g.lower() for g in given], sur.lower()


def name_matches(roster_name, candidate_name):
    """Strict comparison. OpenAlex search is fuzzy: a query for 'Ka Chung Ng' also
    returns 'Ka Wai Ng', 'Ka Po Ng', 'Ka Lok Ng' — different people sharing a surname
    and a first syllable. Every given-name token must therefore be reconciled, not
    just the first and last."""
    rg, rs = _split_name(roster_name)
    cg, cs = _split_name(candidate_name)
    if not rs or not cs or rs != cs:
        return False
    if not rg or not cg:
        return True

    def compat(a, b):
        if a == b:
            return True
        # allow an initial to stand for a full token: "k" vs "ka chung"
        return (len(a) == 1 and b.startswith(a)) or (len(b) == 1 and a.startswith(b))

    small, large = (cg, rg) if len(cg) <= len(rg) else (rg, cg)
    used = []
    for t in small:
        hit = next((u for u in large if u not in used and compat(t, u)), None)
        if hit is None:
            return False                 # a given-name token that cannot be reconciled
        used.append(hit)
    # require at least one full (non-initial) token in common
    return any(len(t) > 1 and t in large for t in small)


def _affil_ids(author):
    """Every institution id appearing anywhere in an author record."""
    ids = set()
    for x in (author.get("affiliations") or []):
        v = _short_id((x or {}).get("institution"))
        if v:
            ids.add(v)
    for x in (author.get("last_known_institutions") or []):
        v = _short_id(x)
        if v:
            ids.add(v)
    return ids


def verify_institution(author, inst_id):
    """Confirm the API filter actually held.

    OpenAlex silently ignores filter keys it does not recognise, returning an
    unfiltered result set that looks legitimate. Every candidate must therefore be
    re-checked against the record itself before it is trusted.

    This only asks "does inst_id appear ANYWHERE in this author's history" — a
    coarse pre-filter to drop junk the API filter let through, not a claim about
    current employment. See institution_match_level() for the recency-aware check
    used to decide trust level.
    """
    return bool(inst_id) and inst_id in _affil_ids(author)


def _affil_year_map(author):
    """institution id -> most recent year this author is recorded there."""
    out = {}
    for x in (author.get("affiliations") or []):
        sid = _short_id((x or {}).get("institution"))
        if not sid:
            continue
        years = [y for y in (x.get("years") or []) if isinstance(y, int)]
        if years:
            out[sid] = max(out.get(sid, 0), max(years))
    return out


def institution_match_level(author, inst_id):
    """Is inst_id this author's CURRENT institution, or just something in their past?

    Roster institutions here are read off official faculty pages — i.e. they are a
    *current* affiliation. An OpenAlex author record instead lists every institution
    ever seen across the author's whole career, so "inst_id appears somewhere" says
    nothing about today: someone who moved on eight years ago still carries the old
    institution forever, and OpenAlex's own author-disambiguation is unreliable
    enough on common names that unrelated people's institutions end up on one ID.
    Recency is the right test, not mere presence.

    Returns "current" | "historical" | "none".
    """
    if not inst_id:
        return "none"
    last_known_ids = {_short_id(i) for i in (author.get("last_known_institutions") or [])}
    last_known_ids.discard(None)
    if inst_id in last_known_ids:
        return "current"
    year_map = _affil_year_map(author)
    if inst_id not in year_map:
        return "none"
    latest_year = max(year_map.values())
    # Tolerance of 1 year: a paper submitted just before a move can still post-date
    # it in OpenAlex's affiliation-year data, so treat "within 1 year of this
    # author's own most recent recorded affiliation" as current rather than
    # requiring an exact tie.
    return "current" if year_map[inst_id] >= latest_year - 1 else "historical"


def _short_id(a):
    """OpenAlex ids are URLs; some records carry a null id. Never assume."""
    v = (a or {}).get("id")
    return v.rsplit("/", 1)[-1] if isinstance(v, str) and v else None


def _full(author, mailto):
    """List endpoints sometimes omit topic fields; fetch the full record when needed."""
    if author.get("topics") or author.get("x_concepts") or author.get("concepts"):
        return author
    sid = _short_id(author)
    if not sid:
        return author
    try:
        return _get(f"authors/{sid}", mailto)
    except Exception:
        return author


def field_score(author, field, mailto=None):
    """Overlap between an author's OpenAlex topics and an expected field."""
    if not field:
        return 0
    a = _full(author, mailto)
    hints = FIELD_HINTS.get(field.lower(), [field.lower()])
    blob = " ".join(t.get("display_name", "") for t in (a.get("topics") or [])).lower()
    blob += " " + " ".join(x.get("display_name", "") for x in (a.get("x_concepts") or [])).lower()
    blob += " " + " ".join(str(c.get("display_name", "")) for c in (a.get("concepts") or [])).lower()
    return sum(1 for h in hints if h in blob)


def _current_institution_names(a, cap_fallback=True):
    """The institution(s) this author is CURRENTLY known for — same source the
    candidate listing shows the user (see _fmt's "last_known" field): OpenAlex's
    own last_known_institutions, falling back to the affiliations list if
    that's empty (which it frequently is).

    cap_fallback=True (the display convention) caps the fallback at the first 3
    entries for readability. Counting how many institutions a candidate has
    must NOT use that cap: if it did, a genuinely messy profile could never
    register more than 3 institutions via the fallback path, and the "noisy"
    side of _clean_single_institution_pick's threshold (>3) would be
    unreachable whenever last_known_institutions happens to be empty — which
    is common. Pass cap_fallback=False when the result feeds a count.
    """
    last = a.get("last_known_institutions") or []
    if last:
        return [x.get("display_name") for x in last if x.get("display_name")]
    affils = (a.get("affiliations") or [])[:3] if cap_fallback else (a.get("affiliations") or [])
    return [(x.get("institution") or {}).get("display_name") for x in affils
            if (x.get("institution") or {}).get("display_name")]


def _clean_single_institution_pick(work_pool, field, mailto):
    """See the call site in _find_author_inner for the reasoning. Returns the
    single clean candidate to pick, or None if the pool doesn't match this
    pattern (more than one clean candidate, no noisy ones to contrast against,
    or the clean candidate's own topics look scattered rather than
    concentrated).

    "Concentrated in the research field" is checked via topic-domain diversity
    (reusing _contamination_risk) rather than a literal keyword hit against
    FIELD_HINTS: a clean candidate's actual topic wording frequently doesn't
    contain any FIELD_HINTS phrase verbatim (field_score()==0 for everyone in
    the pool is common), so a keyword-hit requirement would defeat the
    heuristic on exactly the cases it exists for. A coherent, non-scattered
    topic set is the more reliable signal that these publications belong to
    one real, focused researcher.

    n_inst() counts CURRENT/recent institutions (_current_institution_names,
    uncapped), not the author's full career-long affiliation history
    (_affil_ids): almost everyone has 2+ institutions across a PhD + postdoc +
    current job, so counting the whole history would fail this heuristic for
    nearly any real, uncontaminated professor. What actually distinguishes a
    clean profile from a contaminated one is how many institutions they are
    CURRENTLY listed at.
    """
    def n_inst(a):
        return len(set(_current_institution_names(a, cap_fallback=False)))
    clean = [a for a in work_pool if n_inst(a) <= 1]
    noisy = [a for a in work_pool if n_inst(a) > 3]
    if len(clean) != 1 or not noisy or len(clean) + len(noisy) != len(work_pool):
        return None
    cand = clean[0]
    risk, _ = _contamination_risk(cand, mailto)
    return None if risk else cand


def _find_author_inner(name, institution, mailto=None, field=None, min_works=0):
    """Return (candidates, how). Institution is verified per record, not assumed."""
    inst_id, inst_name = resolve_institution(institution, mailto)
    variants = name_variants(name)
    seen_ids, pooled, rejected = set(), [], 0

    if inst_id:
        for variant in variants:
            for filt in (f"display_name.search:{variant},affiliations.institution.id:{inst_id}",
                         f"display_name.search:{variant},last_known_institutions.id:{inst_id}"):
                try:
                    d = _get("authors", mailto, **{"filter": filt, "per-page": 25})
                except Exception:
                    continue
                for a in d.get("results", []):
                    sid = _short_id(a)
                    if not sid or sid in seen_ids:
                        continue
                    if not verify_institution(a, inst_id):
                        rejected += 1          # filter was ignored by the API
                        continue
                    if not name_matches(name, a.get("display_name")):
                        rejected += 1          # fuzzy search matched a different person
                        continue
                    if not name_compatible(variant, a.get("display_name")):
                        rejected += 1          # fuzzy search matched a different person
                        continue
                    if (a.get("works_count") or 0) < min_works:
                        continue
                    seen_ids.add(sid); pooled.append(a)

        if pooled:
            # Roster institutions come from official faculty pages, i.e. they are
            # CURRENT affiliations. Prefer candidates whose most recent recorded
            # affiliation is this institution over ones where it only appears
            # somewhere in career history — the latter is common noise for authors
            # who moved on, or whose OpenAlex ID has absorbed an unrelated person
            # who once passed through the same institution.
            current = [a for a in pooled if institution_match_level(a, inst_id) == "current"]
            work_pool = current or pooled
            hist_tag = "" if current else "_historical_institution_only"

            if len(work_pool) == 1:
                method = "matched_on_current_institution_verified" if current \
                    else "matched_on_historical_institution_only"
                return _fmt(work_pool, inst_name, inst_id, mailto=mailto), method

            if field:
                scored = [(field_score(x, field, mailto), x) for x in work_pool]
                best = max(sc for sc, _ in scored)
                if best > 0:
                    work_pool = [x for sc, x in scored if sc == best]
            if len(work_pool) == 1:
                method = "matched_on_current_institution_and_field" if current \
                    else "matched_on_historical_institution_only"
                return _fmt(work_pool, inst_name, inst_id, mailto=mailto), method

            # Heuristic: a clean, single-institution profile vs. several
            # noisy, many-institution ones in the same pool. A real person whose
            # OpenAlex record is uncontaminated typically has a short, coherent
            # affiliation list; an ID that has absorbed unrelated same-name
            # people accumulates institutions that don't belong to them. If
            # exactly one candidate has 1 institution on record while every
            # other candidate in the pool has more than 3, AND that clean
            # candidate's own topics look coherent (not scattered across
            # unrelated fields), treat it as the match rather than leaving the
            # whole group ambiguous — the messy candidates are very unlikely to
            # be the person the roster is describing. Note this does NOT
            # require `field` to be set: _clean_single_institution_pick judges
            # "concentrated" via topic-domain coherence (_contamination_risk),
            # not a FIELD_HINTS keyword hit, so it works even when no expected
            # field was given.
            if len(work_pool) > 1:
                picked = _clean_single_institution_pick(work_pool, field, mailto)
                if picked is not None:
                    method = "matched_on_current_institution_clean_profile_heuristic" + hist_tag
                    return _fmt([picked], inst_name, inst_id, mailto=mailto), method

            # Merge only when EVERY pair in the pool looks like one person split
            # across two OpenAlex entities: a split-entity timeline link (each
            # record's most-recent institution appears in the other's affiliation
            # history — the signature of OpenAlex creating a fresh entity when an
            # affiliation was updated rather than editing one in place) PLUS an
            # exact match on fixed identity fields (full name, and ORCID when
            # present). Topic overlap alone is checked too, but only as an extra
            # sanity net — it is not sufficient by itself, since it can also
            # coincidentally line up for two different people in the same field.
            names_ok = all(name_compatible(work_pool[0].get("display_name"),
                                           x.get("display_name")) for x in work_pool[1:])
            if (names_ok and _split_entity_pool(work_pool)
                    and _looks_like_same_person(work_pool, mailto)):
                work_pool = sorted(work_pool, key=lambda a: -(a.get("works_count") or 0))
                return [_merge(work_pool, inst_name, mailto=mailto)], "merged_duplicate_records" + hist_tag
            return _fmt(work_pool, inst_name, inst_id, mailto=mailto), "ambiguous_same_institution" + hist_tag

    for variant in variants:
        try:
            d = _get("authors", mailto,
                     **{"filter": f"display_name.search:{variant}", "per-page": 25})
        except Exception:
            continue
        res = [a for a in d.get("results", [])
               if _short_id(a) and name_compatible(variant, a.get("display_name"))]
        if res:
            return _fmt(res, inst_name), "name_only_unverified"
    return [], "not_found"


def find_author(name, institution, mailto=None, field=None, min_works=0):
    """Wraps _find_author_inner to add one more check that no single return path
    inside it can cover on its own: whether the resolved OpenAlex entity's own
    topics span clearly unrelated fields (see pitfalls.md #4/#13). Institution
    and recency checks only ever compare BETWEEN candidates; a single candidate
    that is itself a polluted OpenAlex entity (absorbing a different real
    person's work under one ID) sails straight through every earlier check
    because there is no second candidate to compare it against.
    """
    cands, how = _find_author_inner(name, institution, mailto, field, min_works)
    if cands and not how.startswith("name_only") and how != "not_found":
        if any(c.get("contamination_risk") for c in cands):
            how = how + "_profile_contamination_risk"
    return cands, how


def _topic_set(author, mailto):
    a = _full(author, mailto)
    out = set()
    for key in ("topics", "x_concepts", "concepts"):
        for t in (a.get(key) or []):
            n = (t or {}).get("display_name")
            if n:
                out.add(n.lower())
    return out


def _most_recent_inst_ids(author):
    """Institution ids at this author's own most recent recorded affiliation year,
    plus anything OpenAlex itself already calls 'last known'."""
    ids = {_short_id(i) for i in (author.get("last_known_institutions") or [])}
    ids.discard(None)
    year_map = _affil_year_map(author)
    if year_map:
        latest = max(year_map.values())
        ids |= {i for i, y in year_map.items() if y >= latest - 1}
    return ids


def _cross_recency_link(a, b):
    """Is this the fingerprint of one person split into two OpenAlex entities?

    The common cause of a real duplicate is: a professor's institution gets
    updated, and OpenAlex creates a NEW author entity for the new affiliation
    rather than editing the old one in place. The two entities then chain
    end-to-end: entity A's most-recent institution is exactly where entity B's
    affiliation history stops (or vice versa). Plain topic overlap cannot tell
    this apart from two different people who simply work in the same field —
    this checks the institution timeline itself lines up instead.
    """
    a_recent, b_recent = _most_recent_inst_ids(a), _most_recent_inst_ids(b)
    a_hist, b_hist = _affil_ids(a), _affil_ids(b)
    return bool(a_recent & b_hist) or bool(b_recent & a_hist)


def _norm_name_exact(name):
    """Token-set identity, stricter than name_compatible(): every token must
    agree, not just surname-plus-initials compatibility. Used as a gate right
    before merging, where a false positive silently doubles someone's output."""
    return tuple(sorted(_tokens(name)))


def _identity_fields_match(a, b):
    """Fixed-field check required on top of a timeline link before two OpenAlex
    entities are trusted as the same person: full name token set must match
    exactly, and if both records carry an ORCID, the ORCIDs must agree. Coarse
    signals like institution recency or topic overlap can coincidentally line up
    for two different people who share a surname; this is the closest identity
    check available from bare author-search records.
    """
    if _norm_name_exact(a.get("display_name")) != _norm_name_exact(b.get("display_name")):
        return False
    oa, ob = a.get("orcid"), b.get("orcid")
    if oa and ob and oa != ob:
        return False
    return True


def _split_entity_pool(records):
    """True only if EVERY pair in the pool looks like the same person recorded
    as two entities: a cross-recency timeline link AND matching fixed identity
    fields for that pair. One mismatching pair fails the whole pool — this
    never merges "most of" a group, since a partial merge is exactly as unsafe
    as a wrong one.
    """
    for i in range(len(records)):
        for j in range(i + 1, len(records)):
            if not (_cross_recency_link(records[i], records[j])
                    and _identity_fields_match(records[i], records[j])):
                return False
    return True


def _looks_like_same_person(records, mailto, min_overlap=0.15):
    """Duplicate entities share research topics; different people usually do not."""
    sets = [_topic_set(r, mailto) for r in records]
    base = sets[0]
    if not base:
        return False
    for other in sets[1:]:
        if not other:
            return False
        inter = len(base & other)
        union = len(base | other) or 1
        if inter / union < min_overlap:
            return False
    return True


# Domains that legitimately co-occur with a non-medical field for one real
# person are common (e.g. CS + Math, Economics + Business); a MEDICAL domain
# sharing an ID with a clearly non-medical one is a much stronger contamination
# signal than domain count alone, since it rarely reflects one person's actual
# combined research interests.
_MEDICAL_DOMAINS = {"Medicine", "Health Professions", "Nursing", "Dentistry",
                    "Veterinary", "Biochemistry, Genetics and Molecular Biology"}


def _topic_domains(a, mailto, min_concept_score=0.35):
    """Level-0 ('domain') classification of this author's own aggregate topics.

    Used to catch a pollution pattern that institution/recency checks cannot see:
    OpenAlex's author disambiguation is unreliable for common names and sometimes
    absorbs a DIFFERENT real person's work into the SAME author ID — this shows
    up as papers spanning clearly unrelated fields on what is supposedly one
    candidate, not two, so no merge-time check ever runs on it.
    """
    full = _full(a, mailto)
    domains = set()
    for t in (full.get("topics") or []):
        d = (t.get("domain") or {}).get("display_name")
        if d:
            domains.add(d)
    for c in (full.get("x_concepts") or []):
        if (c.get("level") == 0) and (c.get("score") or 0) >= min_concept_score:
            d = c.get("display_name")
            if d:
                domains.add(d)
    return domains


def _contamination_risk(a, mailto):
    """(risk: bool, domains seen) for a single already-resolved OpenAlex entity."""
    domains = _topic_domains(a, mailto)
    if not domains:
        return False, domains
    if (domains & _MEDICAL_DOMAINS) and (domains - _MEDICAL_DOMAINS):
        return True, domains
    return len(domains) >= 3, domains


def _merge(records, inst_name, mailto=None):
    """Combine duplicate author entities into one, keeping every id for traceability."""
    primary = records[0]
    out = _fmt([primary], inst_name, mailto=mailto)[0]
    out["works"] = sum(r.get("works_count") or 0 for r in records)
    out["cited_by"] = sum(r.get("cited_by_count") or 0 for r in records)
    out["merged_ids"] = [_short_id(r) for r in records]
    out["merged_names"] = [r.get("display_name") for r in records]
    out["merge_note"] = (f"{len(records)} OpenAlex records were merged as one person: for every "
                         f"pair, one record's most-recent institution appears in the other's "
                         f"affiliation history (consistent with OpenAlex creating a new entity "
                         f"when the affiliation was updated, rather than editing one in place), "
                         f"and full name (+ ORCID, where present) matched exactly. Counts summed. "
                         f"Still worth a spot check if the names differ substantially.")
    return out


def _fmt(res, inst_name, inst_id=None, mailto=None):
    out = []
    for a in res:
        if not _short_id(a):
            continue
        risk, domains = _contamination_risk(a, mailto) if mailto else (None, set())
        out.append({
            "id": _short_id(a),
            "name": a.get("display_name"),
            "works": a.get("works_count"),
            "cited_by": a.get("cited_by_count"),
            "match_level": institution_match_level(a, inst_id) if inst_id else None,
            "last_known": ([i.get("display_name") for i in (a.get("last_known_institutions") or [])]
                           or [(x.get("institution") or {}).get("display_name")
                               for x in (a.get("affiliations") or [])[:3]]),
            "topics": [t.get("display_name") for t in (a.get("topics") or [])[:5]],
            "contamination_risk": risk,
            "contamination_domains": sorted(domains) if domains else [],
            "resolved_institution": inst_name,
        })
    return out


In [ ]:
%%writefile batch_enrich.py
#!/usr/bin/env python3
"""Batch-enrich an entire roster from OpenAlex in one run.

Fills the fields a ranking formula actually needs — recent output, venues,
career start, coauthors — for everyone at once, so that ranking is not biased
by how much manual attention each person happened to receive.

Deliberately does NOT try to resolve every ambiguous name. Unresolved rows are
reported for manual handling rather than guessed at.

Input CSV/XLSX must contain at least: id, name, institution
Usage:
  python batch_enrich.py roster.xlsx --sheet Sheet1 --mailto you@x.com --out enriched.json
  python batch_enrich.py roster.json --mailto you@x.com --out enriched.json --since 2022
"""
import argparse, json, os, sys, time, traceback
from collections import Counter

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import fetch_openalex as oa
import resolve_v2 as rv

CACHE = ".openalex_cache.json"


def load_roster(path, sheet=None, cols=None):
    if path.lower().endswith((".xlsx", ".xlsm")):
        import openpyxl
        wb = openpyxl.load_workbook(path, data_only=True)
        ws = wb[sheet] if sheet else wb.active
        hdr = {str(ws.cell(1, c).value).strip(): c for c in range(1, ws.max_column + 1)
               if ws.cell(1, c).value}
        cmap = cols or {}
        idc = cmap.get("id") or hdr.get("id") or 1
        nmc = cmap.get("name") or hdr.get("name") or 4
        inc = cmap.get("institution") or hdr.get("institution") or 3
        out = []
        for r in range(2, ws.max_row + 1):
            if not ws.cell(r, nmc).value:
                continue
            out.append({"id": ws.cell(r, idc).value,
                        "name": str(ws.cell(r, nmc).value),
                        "institution": str(ws.cell(r, inc).value or "")})
        return out
    return json.load(open(path, encoding="utf-8"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("roster")
    ap.add_argument("--sheet"); ap.add_argument("--mailto")
    ap.add_argument("--out", default="enriched.json")
    ap.add_argument("--since", type=int, help="restrict works to this year onward")
    ap.add_argument("--sleep", type=float, default=0.2)
    ap.add_argument("--field", default="finance",
                    help="expected field, used to break ties: finance/accounting/economics/is/om")
    ap.add_argument("--no-cache", action="store_true",
                    help="ignore .openalex_cache.json and re-fetch everyone")
    a = ap.parse_args()

    cache = ({} if a.no_cache else
             (json.load(open(CACHE, encoding="utf-8")) if os.path.exists(CACHE) else {}))
    roster = load_roster(a.roster, a.sheet)
    print(f"roster: {len(roster)} rows", file=sys.stderr)

    out, stats = [], Counter()
    for i, rec in enumerate(roster, 1):
        # field/since must be part of the key: changing either between runs
        # (e.g. re-running with a different --field, or a different --since
        # window) must NOT silently reuse a result computed under the old
        # settings just because name+institution match.
        key = f"{rec['name']}|{rec['institution']}|{a.field}|{a.since}"
        if key in cache:
            rec.update(cache[key]); stats["cached"] += 1; out.append(rec); continue

        # note: rv.find_author() already strips honorifics/CJK internally
        # (name_variants / _split_name), so no separate cleaning is needed here.
        try:
            cands, how = rv.find_author(rec["name"], rec["institution"], a.mailto, field=a.field)
        except Exception as e:
            rec["enrich_status"] = f"error: {e}"
            rec["error_detail"] = traceback.format_exc()[-600:]
            stats["error"] += 1; out.append(rec)
            print(f"  [{i}/{len(roster)}] ERROR      {rec['name']}  -> {type(e).__name__}: {e}",
                  file=sys.stderr)
            print(f"        {rec['error_detail'].strip().splitlines()[-3]}", file=sys.stderr)
            continue

        rec["match_method"] = how
        # "historical_institution_only" means the roster's (current, official-page)
        # institution only shows up in this author's PAST, never as the most recent
        # one on record — i.e. either the person has since moved, or the OpenAlex
        # identity is not clean. "profile_contamination_risk" means this ONE
        # OpenAlex entity's own topics span clearly unrelated fields (e.g. medicine
        # + computer science on the same ID) — a sign OpenAlex's own author
        # disambiguation has absorbed a different real person's work into this ID.
        # Institution/recency checks cannot catch this (it's not a candidate-vs-
        # candidate comparison), so it is checked separately and never auto-accepted
        # even when it is the only candidate found.
        NEEDS_REVIEW = ("name_only_unverified", "not_found")
        is_historical_only = "_historical_institution_only" in how
        is_contaminated = "_profile_contamination_risk" in how
        if len(cands) != 1 or how in NEEDS_REVIEW or is_historical_only or is_contaminated:
            if is_contaminated:
                rec["enrich_status"] = "possible_move" if is_historical_only else "needs_review_contaminated"
            elif is_historical_only:
                rec["enrich_status"] = "possible_move"
            else:
                rec["enrich_status"] = "ambiguous" if cands else "not_found"
            rec["candidates"] = [{"id": c["id"], "name": c["name"], "works": c["works"],
                                  "last_known": c["last_known"], "topics": c["topics"],
                                  "contamination_domains": c.get("contamination_domains"),
                                  "url": f"https://openalex.org/{c['id']}"}
                                 for c in cands[:5]]
            import urllib.parse as _u
            rec["verify_url"] = ("https://openalex.org/authors?search="
                                 + _u.quote(str(rec.get("name", ""))))
            stats[rec["enrich_status"]] += 1; out.append(rec)
            print(f"  [{i}/{len(roster)}] {rec['enrich_status'].upper():<10} {rec['name']}"
                  f"  ({how}, {len(cands)} cand)", file=sys.stderr)
            continue

        aid = cands[0]["id"]
        merged_info = {k: cands[0][k] for k in ("merged_ids", "merged_names", "merge_note")
                       if k in cands[0]}
        try:
            prof = oa.profile(aid, a.mailto)
            works = oa.works(aid, a.mailto, a.since)
        except Exception as e:
            rec["enrich_status"] = f"error: {e}"
            rec["error_detail"] = traceback.format_exc()[-600:]
            stats["error"] += 1; out.append(rec)
            print(f"  [{i}/{len(roster)}] ERROR      {rec['name']}  -> {type(e).__name__}: {e}",
                  file=sys.stderr)
            for ln in traceback.format_exc().strip().splitlines()[-3:]:
                print(f"        {ln.strip()}", file=sys.stderr)
            continue

        recent = [w for w in works if (w.get("year") or 0) >= (a.since or 0)]
        add = {
            "openalex_id": aid,
            "match_method": how,
            "orcid": prof.get("orcid"),
            "academic_start_year": prof.get("earliest_affiliation_year"),
            "last_known_institutions": prof.get("last_known_institutions"),
            "effective_institutions": prof.get("effective_institutions"),
            "affiliation_institutions": prof.get("affiliation_institutions"),
            "topics": prof.get("topics"),
            "counts_by_year": prof.get("counts_by_year"),
            "cited_by_count": prof.get("cited_by_count"),          # 历史总被引（非近年窗口）
            "total_works_count": prof.get("works_count"),          # OpenAlex 记录的历史发表论文总数
            "recent_works_count": len(recent),                     # since 年份起的发表数
            "recent_works": [{"title": w["title"], "year": w["year"], "venue": w["venue"],
                              "published": w["is_published"], "doi": w["doi"],
                              "url": w.get("url"), "institutions": w.get("institutions")}
                             for w in recent[:12]],
            "works": works,   # 完整列表（含 url/institutions/coauthors），供导出逐篇明细用
            "enrich_status": "ok",
            **merged_info,
        }
        rec.update(add); cache[key] = add
        stats["ok"] += 1; out.append(rec)
        print(f"  [{i}/{len(roster)}] ok         {rec['name']}  "
              f"({len(recent)} recent works, {how})", file=sys.stderr)
        time.sleep(a.sleep)
        if i % 25 == 0:
            json.dump(cache, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)

    json.dump(cache, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(out, open(a.out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print(f"\n{dict(stats)}", file=sys.stderr)
    print(f"wrote {a.out}", file=sys.stderr)
    amb = [(r["name"], r.get("enrich_status")) for r in out if r.get("enrich_status") != "ok"]
    if amb:
        print(f"\n{len(amb)} names need manual resolution (not guessed):", file=sys.stderr)
        for n, st in amb[:30]:
            print(f"  - {n}  [{st}]", file=sys.stderr)


if __name__ == "__main__":
    main()


In [ ]:
import os, json
os.makedirs('assets', exist_ok=True)
open('assets/institution_aliases.json','w',encoding='utf-8').write(r'''{
  "_comment": "Seed aliases. resolve_v2.py appends newly learned mappings here automatically.",
  "_format": "roster_string -> {openalex_id, display_name, learned}",
  "NUS": {"display_name": "National University of Singapore", "learned": false},
  "NTU": {"display_name": "Nanyang Technological University", "learned": false},
  "HKU": {"display_name": "University of Hong Kong", "learned": false},
  "HKUST": {"display_name": "Hong Kong University of Science and Technology", "learned": false},
  "CUHK": {"display_name": "Chinese University of Hong Kong", "learned": false},
  "PolyU": {"display_name": "Hong Kong Polytechnic University", "learned": false},
  "CityUHK": {"display_name": "City University of Hong Kong", "learned": false},
  "UCHI": {"display_name": "University of Chicago", "learned": false},
  "WashU": {"display_name": "Washington University in St. Louis", "learned": false},
  "UMich": {"display_name": "University of Michigan", "learned": false},
  "UCLA": {"display_name": "University of California, Los Angeles", "learned": false},
  "UCB": {"display_name": "University of California, Berkeley", "learned": false},
  "UPenn": {"display_name": "University of Pennsylvania", "learned": false},
  "UT Austin": {"display_name": "University of Texas at Austin", "learned": false},
  "CMU": {"display_name": "Carnegie Mellon University", "learned": false},
  "LSE": {"display_name": "London School of Economics and Political Science", "learned": false},
  "LBS": {"display_name": "London Business School", "learned": false},
  "SMU": {"display_name": "Singapore Management University", "learned": false},
  "KAIST": {"display_name": "Korea Advanced Institute of Science and Technology", "learned": false},
  "PKU": {"display_name": "Peking University", "learned": false},
  "THU": {"display_name": "Tsinghua University", "learned": false},
  "SJTU": {"display_name": "Shanghai Jiao Tong University", "learned": false},
  "Fudan": {"display_name": "Fudan University", "learned": false},
  "IIT": {"display_name": "Indian Institute of Technology", "learned": false},
  "IIM": {"display_name": "Indian Institute of Management", "learned": false}
}
''')
print('机构别名表已就位（会自动学习新缩写）')

In [ ]:
%%writefile merge_to_excel.py
#!/usr/bin/env python3
"""Write enrichment results into a NEW workbook. The original is never modified.

Appends columns rather than overwriting existing ones, so manually curated
assessments survive. Every appended column is prefixed so provenance is obvious.

Also writes a second sheet, "论文明细", with ONE ROW PER PAPER (title, this
author's institution on that specific paper, a link, and coauthors) — a roster
row only has room for a compact preview of recent work, but downstream use
(coauthor network analysis, checking a specific claim) needs the full list.

Usage:
  python merge_to_excel.py original.xlsx enriched.json --sheet 全部教授_论文与聚焦度 \
      --out enriched_output.xlsx
"""
import argparse, json, re, shutil, os, sys
from datetime import date
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import resolve_v2 as rv   # reuse the SAME alias table / resolver batch_enrich.py
                          # already used to find these candidates, instead of
                          # reinventing a weaker raw-substring check here.

# NOTE: "近年" here means "since whatever --since year batch_enrich.py was run
# with" (see recent_works_count / recent_works in the enriched JSON) — it is
# NOT hardcoded to a fixed 3-year window, despite what an earlier version of
# this column's name implied.
NEW_COLS = [
    ("OA_匹配状态", "enrich_status"),
    ("OA_匹配方式", "match_method"),
    ("OA_机构时效性", None),          # current(现任) / historical(仅历史上出现过) / — (无年份数据)
    ("OA_作者ID", "openalex_id"),
    ("OA_ORCID", "orcid"),
    ("OA_当前机构", None),
    ("OA_任教起始年", "academic_start_year"),
    ("OA_被引总数(历史总计)", "cited_by_count"),
    ("OA_发表论文总数(历史总计)", "total_works_count"),
    ("OA_近年发表数", None),
    ("OA_近年正式发表数", None),
    ("OA_最新发表年", None),
    ("OA_近年论文清单(标题/机构/链接见「论文明细」表)", None),
    ("OA_研究主题", None),
    ("OA_机构是否不一致", None),
    ("OA_候选人(需人工确认)", None),
    ("OA_合并记录说明", None),
    ("OA_可信度分", None),
    ("OA_可信度等级", None),
    ("OA_建议动作", None),
    ("OA_可信度依据", None),
]
WRAP_COLS = (12, 15, 16, 20)   # 0-based positions in vals[] that need wrap_text
WIDE_COLS = {12: 60, 15: 45, 20: 70}

PAPER_COLS = ["教授姓名", "名单机构", "论文标题", "发表年份", "期刊/会议",
             "该作者当次所属机构", "文章链接", "DOI", "合作者"]


def _norm_inst(s):
    return re.sub(r"[^a-z0-9]", "", str(s or "").lower())


def _institution_mismatch(inst_sheet, last, mailto=None):
    """Does `last` (this author's OpenAlex institution names) actually include
    the roster's institution?

    A raw substring test on the roster's raw text fails whenever the roster
    uses an acronym ("HKU") that never appears verbatim inside OpenAlex's full
    canonical name ("University of Hong Kong") — that check would flag a
    correct match as a mismatch. Resolve the roster string through the SAME
    alias table / resolver used to find this candidate in the first place
    (resolve_v2.resolve_institution — checks the local alias file before ever
    touching the network, and since batch_enrich.py already resolved every
    institution in this roster, this almost always hits that local cache).
    """
    if not inst_sheet or not last:
        return False
    try:
        _, canonical = rv.resolve_institution(inst_sheet, mailto)
    except Exception:
        canonical = None
    targets = {_norm_inst(canonical)} if canonical else set()
    targets.add(_norm_inst(inst_sheet))       # also allow a literal/raw match
    targets.discard("")
    for x in last:
        nx = _norm_inst(x)
        if not nx:
            continue
        if nx in targets or any(t and (t in nx or nx in t) for t in targets):
            return False    # a match was found — NOT a mismatch
    return True


def _write_roster_sheet(wb, ws, data, a):
    norm = lambda s: re.sub(r"\s+", "", str(s or "")).lower()
    # Key on (name, institution), not name alone: a roster with two different
    # people who share a name (common with frequent Chinese surnames) would
    # otherwise have the later one's enrichment silently overwrite the
    # earlier one's in this dict, and BOTH rows would then be merged with
    # whichever one happened to load last.
    by_name_inst = {(norm(r.get("name")), norm(r.get("institution"))): r for r in data}
    # Fallback for the (rare) case a name doesn't reduce to a unique
    # (name, institution) pair here but is otherwise unambiguous in `data`.
    by_name_only = {}
    for r in data:
        k = norm(r.get("name"))
        by_name_only[k] = None if k in by_name_only else r
    start = ws.max_column + 1
    BASE = Font(name="微软雅黑", size=9)
    for j, (title, _) in enumerate(NEW_COLS):
        c = ws.cell(1, start + j)
        c.value = title
        c.font = Font(name="微软雅黑", size=9, bold=True)
        c.fill = PatternFill("solid", fgColor="FFE7F0D9")
    for j, w in WIDE_COLS.items():
        ws.column_dimensions[openpyxl.utils.get_column_letter(start + j)].width = w

    n_ok = n_amb = n_moved = n_contaminated = 0
    for r in range(2, ws.max_row + 1):
        nm = ws.cell(r, a.name_col).value
        if not nm:
            continue
        inst_sheet_raw = str(ws.cell(r, a.inst_col).value or "")
        rec = by_name_inst.get((norm(nm), norm(inst_sheet_raw))) or by_name_only.get(norm(nm))
        if not rec:
            continue
        works = rec.get("recent_works") or []
        years = [w["year"] for w in works if w.get("year")]
        inst_sheet = inst_sheet_raw
        # effective_institutions already falls back to affiliation history when
        # OpenAlex's own last_known_institutions is empty (which it frequently is,
        # even for authors with a full affiliation record) — using the raw
        # last_known_institutions field alone silently shows "[]" for such rows.
        last = (rec.get("effective_institutions") or rec.get("last_known_institutions")
                or rec.get("affiliation_institutions") or [])
        method = str(rec.get("match_method") or "")
        if "_historical_institution_only" in method:
            timeliness = "historical（仅历史上出现过，可能已转校，需核实）"
        elif "current" in method:
            timeliness = "current（最近一次隶属）"
        elif method in ("name_only_unverified", "not_found"):
            timeliness = "—"
        else:
            timeliness = "unknown（无机构年份数据，无法判断时效性）"
        if "_profile_contamination_risk" in method:
            timeliness += " ⚠️疑似身份污染（该ID论文跨越不相关学科，可能混入他人成果）"
        mismatch = ""
        if rec.get("enrich_status") == "ok" and last and inst_sheet:
            if _institution_mismatch(inst_sheet, last, a.mailto):
                mismatch = "⚠ 需核实"
                n_moved += 1
        rel = rec.get("reliability") or {}
        vals = [
            rec.get("enrich_status"),
            rec.get("match_method"),
            timeliness,
            rec.get("openalex_id"),
            rec.get("orcid"),
            "；".join(map(str, last)),
            rec.get("academic_start_year"),
            rec.get("cited_by_count"),
            rec.get("total_works_count"),
            rec.get("recent_works_count", len(works)),
            sum(1 for w in works if w.get("published")),
            max(years) if years else None,
            " || ".join(f"{w.get('year')} {w.get('venue') or '—'} | {str(w.get('title'))[:60]}"
                        f"{' | ' + '/'.join(w.get('institutions') or []) if w.get('institutions') else ''}"
                        for w in works[:8]),
            "；".join((rec.get("topics") or [])[:6]),
            mismatch,
            "；".join(f"{c.get('name')}({c.get('works')}篇,{'/'.join(map(str,c.get('last_known') or []))[:28]})"
                     for c in (rec.get("candidates") or [])[:4]),
            ("合并了 " + " + ".join(rec.get("merged_names") or []) + f"（共{len(rec.get('merged_ids') or [])}条记录）")
            if rec.get("merged_ids") else "",
            rel.get("confidence"),
            rel.get("level"),
            rel.get("action"),
            "；".join((rel.get("reasons") or []) + (rel.get("flags") or [])),
        ]
        for j, v in enumerate(vals):
            cell = ws.cell(r, start + j)
            cell.value = v
            cell.font = BASE
            cell.alignment = Alignment(wrap_text=(j in WRAP_COLS), vertical="top")
        if rec.get("enrich_status") == "ok":
            n_ok += 1
        elif rec.get("enrich_status"):
            # Anything that isn't "ok" needs a human look — count it here rather
            # than maintaining a hardcoded status list that drifts out of sync
            # with batch_enrich.py every time a new status is added.
            n_amb += 1
            if rec.get("enrich_status") == "needs_review_contaminated":
                n_contaminated += 1
    return n_ok, n_amb, n_moved, n_contaminated


def _write_papers_sheet(wb, data, a):
    """One row per paper across every enriched roster member: title, this
    author's own institution ON that paper, a link, and coauthors — the detail
    a single roster row has no room for.
    """
    ws2 = wb.create_sheet("论文明细")
    BASE = Font(name="微软雅黑", size=9)
    for j, title in enumerate(PAPER_COLS, start=1):
        c = ws2.cell(1, j)
        c.value = title
        c.font = Font(name="微软雅黑", size=9, bold=True)
        c.fill = PatternFill("solid", fgColor="FFE7F0D9")
    for col, width in ((3, 55), (5, 30), (6, 30), (7, 40), (9, 45)):
        ws2.column_dimensions[openpyxl.utils.get_column_letter(col)].width = width

    r = 2
    n_papers = 0
    for rec in data:
        works = rec.get("works")
        if not works:
            continue
        roster_inst = rec.get("institution") or ""
        for w in works:
            coauthors = "；".join(c.get("name") for c in (w.get("coauthors") or [])
                                 if c.get("name"))
            row = [
                rec.get("name"),
                roster_inst,
                w.get("title"),
                w.get("year"),
                w.get("venue"),
                "；".join(w.get("institutions") or []),
                w.get("url"),
                w.get("doi"),
                coauthors,
            ]
            for j, v in enumerate(row, start=1):
                cell = ws2.cell(r, j)
                cell.value = v
                cell.font = BASE
                cell.alignment = Alignment(wrap_text=(j in (3, 9)), vertical="top")
            r += 1
            n_papers += 1
    return n_papers


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("original"); ap.add_argument("enriched")
    ap.add_argument("--sheet"); ap.add_argument("--out")
    ap.add_argument("--name-col", type=int, default=4)
    ap.add_argument("--inst-col", type=int, default=3)
    ap.add_argument("--mailto", help="only used if an institution isn't already "
                                     "in the local alias cache from batch_enrich.py")
    a = ap.parse_args()

    out = a.out or f"{os.path.splitext(a.original)[0]}_OA补全_{date.today().isoformat()}.xlsx"
    if os.path.abspath(out) == os.path.abspath(a.original):
        raise SystemExit("refusing to overwrite the original file")
    shutil.copy(a.original, out)          # work on a copy; original untouched

    wb = openpyxl.load_workbook(out)
    ws = wb[a.sheet] if a.sheet and a.sheet in wb.sheetnames else wb.active
    data = json.load(open(a.enriched, encoding="utf-8"))

    n_ok, n_amb, n_moved, n_contaminated = _write_roster_sheet(wb, ws, data, a)
    n_papers = _write_papers_sheet(wb, data, a)

    wb.save(out)
    print(f"原文件未改动：{a.original}")
    print(f"新文件已生成：{out}")
    print(f"  成功匹配 {n_ok} 行｜待人工确认 {n_amb} 行｜机构可能不一致 {n_moved} 行"
         f"｜疑似 OpenAlex 身份污染 {n_contaminated} 行")
    print(f"  新增 {len(NEW_COLS)} 列（均以 OA_ 开头，原有列未被覆盖）")
    print(f"  另新增「论文明细」工作表，共 {n_papers} 篇论文（标题/发文机构/链接/合作者）")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile openalex_links.py
#!/usr/bin/env python3
"""Generate browser URLs for manual verification on OpenAlex.

The web UI is hard to drive by hand for this task; these URLs land directly on a
filtered author list so a human can confirm an identity in seconds.

Usage:
  python openalex_links.py --name "Yingying Li" --institution HKUST
  python openalex_links.py --roster enriched.json > links.md              # all non-ok rows
  python openalex_links.py --roster enriched.json --status possible_move  # one status only
"""
import argparse, json, sys, urllib.parse
sys.path.insert(0, __file__.rsplit("/", 1)[0])
import resolve_v2 as rv


def links(name, institution, mailto=None):
    inst_id, inst_name = rv.resolve_institution(institution, mailto)
    variants = rv.name_variants(name)
    q = urllib.parse.quote(variants[0] if variants else name)
    out = {"name": name, "institution": institution,
           "resolved_institution": inst_name, "institution_id": inst_id,
           "name_variants": variants}
    if inst_id:
        out["authors_at_institution"] = (
            f"https://openalex.org/authors?page=1&filter=display_name.search%3A{q},"
            f"affiliations.institution.id%3A{inst_id}")
        out["all_authors_at_institution"] = (
            f"https://openalex.org/institutions/{inst_id}")
        out["api_check"] = (
            f"https://api.openalex.org/authors?filter=display_name.search:{q},"
            f"affiliations.institution.id:{inst_id}")
    out["name_only"] = f"https://openalex.org/authors?page=1&filter=display_name.search%3A{q}"
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name"); ap.add_argument("--institution")
    ap.add_argument("--roster")
    ap.add_argument("--status", help="filter to one status/method prefix "
                                     "(e.g. possible_move); omit for ALL non-ok rows")
    ap.add_argument("--mailto")
    a = ap.parse_args()
    if a.roster:
        data = json.load(open(a.roster, encoding="utf-8"))
        if a.status:
            # explicit filter: one status or match_method prefix only
            rows = [r for r in data if str(r.get("enrich_status", "")).startswith(a.status)
                    or str(r.get("match_method", "")).startswith(a.status)]
        else:
            # default: everything that isn't "ok" needs a human look — matches the
            # same "not ok" rule merge_to_excel.py uses, so this report and the
            # xlsx's "待人工确认" count always agree on how many rows that is.
            rows = [r for r in data if r.get("enrich_status") != "ok"]
        print(f"# 待人工确认 {len(rows)} 位\n")
        for r in rows:
            L = links(r.get("name"), r.get("institution"), a.mailto)
            print(f"## {r.get('name')}  ({r.get('institution')})  "
                 f"[{r.get('enrich_status')} / {r.get('match_method')}]")
            print(f"- 解析到的机构：{L.get('resolved_institution')}  `{L.get('institution_id')}`")
            print(f"- 尝试过的姓名：{', '.join(L['name_variants'])}")
            if L.get("authors_at_institution"):
                print(f"- **在该机构内按姓名查**：{L['authors_at_institution']}")
            print(f"- 仅按姓名查（会有同名他人）：{L['name_only']}")
            for c in (r.get("candidates") or [])[:5]:
                print(f"    - 候选 `{c.get('id')}` {c.get('name')} | {c.get('works')}篇 "
                      f"| {c.get('last_known')} | {(c.get('topics') or [])[:3]}")
            print()
    else:
        print(json.dumps(links(a.name, a.institution, a.mailto), ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile confidence.py
#!/usr/bin/env python3
"""Score how much each enrichment result should be trusted, with stated reasons.

Deliberately deterministic rather than model-based: the checks are cheap, auditable,
and reproducible, and a language model adds nothing to questions like "does the
institution in the record match the institution in the roster".

Usage: python confidence.py enriched.json --out scored.json
"""
import argparse, json, os, re, sys
from collections import Counter
from datetime import date

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

try:
    from resolve_v2 import same_institution, norm_inst
except Exception:                                   # standalone fallback
    def norm_inst(name):
        n = re.sub(r"[^\w\s]", " ", str(name or "").lower())
        return re.sub(r"^the ", "", re.sub(r"\s+", " ", n).strip())

    def same_institution(a, b):
        x, y = norm_inst(a), norm_inst(b)
        return bool(x and y and (x == y or x in y or y in x))


# Base trust by how identity was resolved. "current_institution" methods mean the
# author's MOST RECENT recorded affiliation is the roster institution (see
# resolve_v2.institution_match_level) — the strongest evidence available given that
# roster institutions are read off official, current faculty pages. "historical"
# methods mean the roster institution only ever appears in the author's past, which
# is why they score low and are always routed to manual review upstream regardless.
# Every key here is a literal string resolve_v2.find_author() can actually return —
# keep this list in sync with that function rather than accumulating entries for
# retired return values.
METHOD_BASE = {
    "matched_on_current_institution_verified": 95,
    "matched_on_current_institution_and_field": 90,
    "matched_on_current_institution_clean_profile_heuristic": 82,
    "merged_duplicate_records": 65,
    "matched_on_historical_institution_only": 30,
    "merged_duplicate_records_historical_institution_only": 25,
    "ambiguous_same_institution": 15,
    "ambiguous_same_institution_historical_institution_only": 10,
    "name_only_unverified": 12,
    "not_found": 0,
}

# Keyword flag for the "contaminated profile" pitfall (see pitfalls.md #4): topics or
# venues from clearly unrelated fields (medicine, physics, materials science, etc.)
# showing up on a supposedly finance/econ/business scholar's recent work.
OFF_FIELD = re.compile(
    r"\b(clinical|cardiolog\w*|oncolog\w*|radiolog\w*|tumou?r|carcinoma|cancer|"
    r"diabetes|neutrino|collider|astrophys\w*|photovoltaic|semiconductor|"
    r"nanoparticle|drug delivery|pharmacolog\w*|biolog\w*|genomic\w*|proteomic\w*|"
    r"immunolog\w*|surgical|pathogen\w*|virus|vaccine)\b", re.I)


def _inst_agrees(roster_inst, record_insts):
    """Institution names are unique, so compare the expanded name exactly.

    Similarity scoring is the wrong tool here: 'Hong Kong University of Science and
    Technology' and 'University of Science and Technology of China' overlap heavily
    while denoting different places. Initials are used earlier to *resolve* an
    abbreviation, never to *verify* one — 'CUHK' is the initialism of both the
    Chinese University and the City University of Hong Kong.
    """
    if not roster_inst or not record_insts:
        return None
    return any(same_institution(roster_inst, r) for r in record_insts)


def score_one(rec, expect_field="finance"):
    pts, why, flags = 0, [], []
    method = rec.get("match_method") or ""
    # "_profile_contamination_risk" can be appended on top of any base method (see
    # resolve_v2.find_author) — strip it for the base lookup, then apply its own
    # penalty once, rather than enumerating every combined string.
    base_method = method.replace("_profile_contamination_risk", "")
    base = METHOD_BASE.get(base_method, 25)
    pts += base
    why.append(f"匹配方式「{base_method or '未知'}」基准 {base} 分")
    if "_profile_contamination_risk" in method:
        pts -= 35
        flags.append("profile_contamination_risk")
        why.append("该 OpenAlex ID 自身的论文跨越明显不相关学科（如医学+计算机）−35"
                   "，很可能是 OpenAlex 消歧错误、混入了另一个同名人的成果，务必人工核实")

    insts = (rec.get("effective_institutions") or rec.get("last_known_institutions")
             or rec.get("affiliation_institutions") or [])
    roster_inst = str(rec.get("institution") or "")
    agree = _inst_agrees(roster_inst, insts)
    if agree is True:
        pts += 10; why.append(f"机构与名单一致 +10（{insts[0]}）")
    elif agree is False:
        pts -= 15
        why.append(f"机构与名单不一致 −15（名单「{roster_inst}」vs 记录「{insts[:2]}」）")
        flags.append("institution_mismatch")
    else:
        pts -= 5; why.append("记录中无任何机构信息 −5")

    merged = rec.get("merged_ids") or []
    if merged:
        n = len(merged)
        names = rec.get("merged_names") or []
        uniq = len({str(x).lower() for x in names})
        if n >= 5:
            pts -= 30; flags.append("over_merge")
            why.append(f"合并了 {n} 条记录 −30（数量过多，很可能混入了他人）")
        elif uniq > 1:
            pts -= 10
            why.append(f"合并了 {n} 条记录、{uniq} 种写法 −10（请核对是否同一人）")
        else:
            why.append(f"合并了 {n} 条同名记录（写法一致，风险较低）")

    works = rec.get("recent_works") or []
    if works:
        venues = " ".join(str(w.get("venue") or "") for w in works)
        titles = " ".join(str(w.get("title") or "") for w in works)
        off = OFF_FIELD.findall(venues + " " + titles)
        if off:
            pts -= 25; flags.append("off_field_papers")
            why.append(f"论文含明显跨领域内容 −25（{sorted(set(x.lower() for x in off))[:3]}）")
        yrs = [w.get("year") for w in works if w.get("year")]
        if yrs and max(yrs) >= date.today().year - 2:
            pts += 5; why.append("含近年论文 +5")
    else:
        pts -= 10; why.append("未取到任何近三年论文 −10")

    if rec.get("orcid"):
        pts += 5; why.append("有 ORCID +5")

    pts = max(0, min(100, pts))
    level = ("高" if pts >= 80 else "中" if pts >= 55 else "低" if pts >= 30 else "不可用")
    action = {
        "高": "可直接采用",
        "中": "建议抽查一眼论文与机构",
        "低": "必须人工确认后再用",
        "不可用": "不要采用，请人工查证",
    }[level]
    return {"confidence": pts, "level": level, "action": action,
            "reasons": why, "flags": flags}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("enriched"); ap.add_argument("--out", default="scored.json")
    ap.add_argument("--field", default="finance")
    a = ap.parse_args()
    data = json.load(open(a.enriched, encoding="utf-8"))
    for r in data:
        r["reliability"] = score_one(r, a.field)
    json.dump(data, open(a.out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    c = Counter(r["reliability"]["level"] for r in data)
    print("=== 可信度分布 ===")
    for lv in ("高", "中", "低", "不可用"):
        if c.get(lv):
            print(f"  {lv}: {c[lv]} 位")
    print("\n=== 需要人工处理的（低 / 不可用）===")
    for r in sorted(data, key=lambda x: x["reliability"]["confidence"]):
        rel = r["reliability"]
        if rel["level"] in ("低", "不可用"):
            print(f"\n  [{rel['confidence']}分·{rel['level']}] {r.get('name')} ({r.get('institution')})")
            print(f"     → {rel['action']}")
            for w in rel["reasons"]:
                print(f"       · {w}")
    print(f"\nwrote {a.out}")


if __name__ == "__main__":
    main()


In [ ]:
import importlib, sys
sys.path.insert(0, '.')
import confidence as _cf
importlib.reload(_cf)
assert hasattr(_cf, 'METHOD_BASE') and hasattr(_cf, 'OFF_FIELD'), 'confidence.py 缺少必要定义'
_test = _cf.score_one({"match_method": "matched_on_current_institution_verified",
                        "effective_institutions": ["测试机构"], "institution": "测试机构",
                        "recent_works": []})
print('confidence.py 自检通过，可以正常打分：', _test["confidence"], _test["level"])


In [ ]:
%%writefile reverse_lookup.py
#!/usr/bin/env python3
"""Reverse lookup: find a person by narrowing the pool first, then matching the name.

Searching a common name globally and then filtering by institution is unreliable —
"Yingying Li" returns medical researchers, physicists and economists, and any of
them can survive a loose institution filter.

Inverting the query is far more robust: list the authors who actually published
from this institution in this field, then look for the name inside that small pool.

Usage:
  python reverse_lookup.py --institution HKUST --field finance --mailto you@x.com
  python reverse_lookup.py --institution HKUST --field finance --name "Yingying Li"
"""
import argparse, json, sys, os
from collections import defaultdict

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import resolve_v2 as rv

# Single source of truth for field keywords lives in resolve_v2.FIELD_HINTS —
# batch_enrich.py's field-tie-break and this reverse lookup must agree on what
# "finance" or "om" means, or the two tools would silently disagree about the
# same person.
FIELD_TOPICS = rv.FIELD_HINTS


def authors_at(institution, field, mailto=None, years=6, max_pages=8):
    """Authors with recent publications from this institution in this field."""
    inst_id, inst_name = rv.resolve_institution(institution, mailto)
    if not inst_id:
        return None, None, {}
    from datetime import date
    since = date.today().year - years
    topics = FIELD_TOPICS.get((field or "").lower(), [field or ""])
    people = defaultdict(lambda: {"name": None, "works": 0, "titles": [], "id": None})

    for topic in topics:
        cursor = "*"
        for _ in range(max_pages):
            try:
                d = rv._get("works", mailto, **{
                    "filter": (f"authorships.institutions.lineage:{inst_id},"
                               f"from_publication_date:{since}-01-01,"
                               f"default.search:{topic}"),
                    "per-page": 200, "cursor": cursor,
                    "select": "id,display_name,publication_year,authorships"})
            except Exception:
                break
            for w in d.get("results", []):
                for au in w.get("authorships") or []:
                    a = au.get("author") or {}
                    sid = rv._short_id(a)
                    if not sid:
                        continue
                    inst_hit = any(rv._short_id(i) == inst_id
                                   for i in (au.get("institutions") or []))
                    if not inst_hit:
                        continue
                    p = people[sid]
                    p["id"] = sid
                    p["name"] = a.get("display_name")
                    p["works"] += 1
                    if len(p["titles"]) < 3:
                        p["titles"].append(f"{w.get('publication_year')} {w.get('display_name')}")
            cursor = (d.get("meta") or {}).get("next_cursor")
            if not cursor:
                break
    return inst_id, inst_name, people


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--institution", required=True)
    ap.add_argument("--field", default="finance")
    ap.add_argument("--name", help="optional: highlight matches for this person")
    ap.add_argument("--mailto"); ap.add_argument("--years", type=int, default=6)
    ap.add_argument("--out")
    a = ap.parse_args()

    inst_id, inst_name, people = authors_at(a.institution, a.field, a.mailto, a.years)
    if not inst_id:
        print(f"could not resolve institution: {a.institution}"); return
    ranked = sorted(people.values(), key=lambda p: -p["works"])
    print(f"{inst_name} ({inst_id}) — {len(ranked)} authors publishing in "
          f"'{a.field}' in the last {a.years} years\n")

    if a.name:
        toks = {t.lower() for t in rv.name_variants(a.name)[0].split()} if rv.name_variants(a.name) else set()
        hits = [p for p in ranked
                if toks and toks <= {w.lower() for w in str(p["name"]).split()}]
        loose = [p for p in ranked
                 if p not in hits and toks & {w.lower() for w in str(p["name"]).split()}]
        print(f"== exact name match: {len(hits)} ==")
        for p in hits:
            print(f"  {p['id']}  {p['name']}  ({p['works']} works here)")
            for t in p["titles"]:
                print(f"      {t[:88]}")
        if loose:
            print(f"\n== partial match: {len(loose)} ==")
            for p in loose[:10]:
                print(f"  {p['id']}  {p['name']}  ({p['works']} works here)")
        if not hits and not loose:
            print("  none — this person may publish under a different romanisation, "
                  "may not be indexed, or may not publish in this field.")
    else:
        for p in ranked[:60]:
            print(f"  {p['id']}  {str(p['name'])[:38]:<40} {p['works']} works")

    if a.out:
        json.dump(ranked, open(a.out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        print(f"\nwrote {a.out}")


if __name__ == "__main__":
    main()


---
## 环境自检（报 FileNotFoundError 时先跑这一格）

In [ ]:
# ===== 环境自检：确认前面的步骤都跑过了 =====
import os
need = {
    "batch_enrich.py":      "第4步 载入脚本",
    "resolve_v2.py":        "第4步 载入脚本",
    "fetch_openalex.py":    "第4步 载入脚本",
    "merge_to_excel.py":    "第4步 载入脚本",
    "assets/institution_aliases.json": "第4步 机构别名表",
}
missing = [(f, s) for f, s in need.items() if not os.path.exists(f)]
try:
    XLSX
    has_xlsx = os.path.exists(XLSX)
except NameError:
    has_xlsx = False
    XLSX = None

print("=== 环境自检 ===")
for f, s in need.items():
    print(f"  {'✅' if os.path.exists(f) else '❌'}  {f:<36} ({s})")
print(f"  {'✅' if has_xlsx else '❌'}  上传的 Excel：{XLSX}   (第2步)")

if missing or not has_xlsx:
    print("\n⚠️ 有缺失。请回到顶部，从第 1 格开始依次运行，")
    print("   或用菜单：代码执行程序 → 全部运行。")
else:
    print("\n✅ 一切就绪，可以继续。")

# 额外诊断：确认脚本能找到机构别名表
try:
    import importlib, sys
    sys.path.insert(0, ".")
    import resolve_v2 as rv
    importlib.reload(rv)
    print(f"\n机构别名表路径：{rv._ALIAS_PATH}")
    print(f"载入别名条目：{len(rv.ALIASES)} 条", "✅" if rv.ALIASES else "❌ 未载入，缩写可能匹配不上")
except Exception as e:
    print(f"\n❌ resolve_v2 载入失败：{e}")

---
## 第 5 步：先小样本试跑（重要）

**先跑 10 个人**，确认列名对得上、能取到数据，再跑全量。
如果这一步报错或者取到的名字明显不对，先别跑全量，把报错信息发给 Claude。

In [ ]:
import openpyxl, json
wb = openpyxl.load_workbook(XLSX, data_only=True)
ws = wb[SHEET] if SHEET in wb.sheetnames else wb.active
print('工作表：', ws.title, '｜行数：', ws.max_row, '｜列数：', ws.max_column)
print('\n前3行读到的内容（确认 id / 姓名 / 学校 是否正确）：')
for r in range(2, 5):
    print(f'  id={ws.cell(r,1).value} | 姓名={ws.cell(r,4).value} | 学校={ws.cell(r,3).value}')

# 截取前10行另存，用于试跑
wb2 = openpyxl.Workbook(); ws2 = wb2.active; ws2.title = 'test'
for c in range(1, ws.max_column+1):
    ws2.cell(1, c).value = ws.cell(1, c).value
for i, r in enumerate(range(2, 12), start=2):
    for c in range(1, ws.max_column+1):
        ws2.cell(i, c).value = ws.cell(r, c).value
wb2.save('test10.xlsx')
print('\n已生成 test10.xlsx（前10人）')

In [ ]:
import subprocess, os, sys
cmd = [sys.executable, "batch_enrich.py", "test10.xlsx", "--sheet", "test",
       "--mailto", MAILTO, "--since", str(SINCE), "--field", FIELD, "--out", "test10.json"]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout)
print(p.stderr)
if p.returncode != 0 or not os.path.exists("test10.json"):
    print("=" * 60)
    print("❌ 脚本执行失败，未生成 test10.json。")
    print("   请把上面这段完整输出（尤其是最后的报错）发给 Claude。")
else:
    print("=" * 60)
    print("✅ test10.json 已生成，可以继续下一格。")

In [ ]:
import json
d = json.load(open('test10.json', encoding='utf-8'))
from collections import Counter
print('状态分布：', dict(Counter(x.get('enrich_status') for x in d)), '\n')
for x in d:
    st = x.get('enrich_status')
    if st == 'ok':
        print(f"✅ {x['name']} ({x['institution']})  [{x.get('match_method')}]")
        print(f"    OpenAlex {x.get('openalex_id')} | 任教起始 {x.get('academic_start_year')} | 被引 {x.get('cited_by_count')}")
        # 用 effective_institutions（有兜底）而不是原始 last_known_institutions
        # （OpenAlex 的 last_known_institutions 经常为空，即使 affiliation 历史是有数据的）
        cur = x.get('effective_institutions') or x.get('last_known_institutions') or []
        print(f"    当前机构：{cur}")
        if x.get('merged_ids'):
            print(f"    🔗 合并了 {len(x['merged_ids'])} 条记录：{x.get('merged_names')}")
        for w in (x.get('recent_works') or [])[:3]:
            print(f"      - {w['year']} {str(w['venue'])[:42]} | {str(w['title'])[:52]}")
    elif st == 'possible_move':
        # 新状态：目标机构只出现在该作者的历史记录里，从未是最近一次隶属 —— 大概率已经转校，
        # 或官网信息与 OpenAlex 记录的时间对不上，需要人工核实，不会被当成 ok 自动采信
        print(f"🚚 {x['name']} ({x['institution']})  [{x.get('match_method')}]  <- 疑似已转校/需核实")
        cur = x.get('effective_institutions') or x.get('last_known_institutions') or []
        print(f"    OpenAlex 显示当前机构：{cur}（名单记的是「{x['institution']}」）")
    elif st in ('ambiguous', 'not_found'):
        print(f"❓ {x['name']} ({x['institution']})  [{x.get('match_method')}]")
        for c2 in (x.get('candidates') or [])[:5]:
            print(f"      候选：{c2.get('name')} | {c2.get('works')}篇 | 机构={c2.get('last_known')}")
    else:
        print(f"❌ {x['name']}  {st}")
    print()


**检查上面的输出**：名字、机构、论文是不是对得上本人？

- ✅ 对 → 继续第 6 步
- ❌ 不对（比如取到了同名的另一个人）→ 先别跑全量，把输出发给 Claude

---
## 第 6 步：跑全量

约 15–25 分钟。中间会实时打印进度。

In [ ]:
import subprocess, os, sys
cmd = [sys.executable, "batch_enrich.py", XLSX, "--sheet", SHEET,
       "--mailto", MAILTO, "--since", str(SINCE), "--field", FIELD, "--out", "enriched.json"]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-4000:])
print(p.stderr[-8000:])
print("=" * 60)
print("✅ 完成" if os.path.exists("enriched.json") else "❌ 未生成 enriched.json，请把上面输出发给 Claude")

---
## 第 7 步：查看结果并下载

In [ ]:
import json
from collections import Counter
d = json.load(open('enriched.json', encoding='utf-8'))
c = Counter(x.get('enrich_status') for x in d)
print('总人数：', len(d))
print('结果分布：', dict(c))

amb = [x['name'] for x in d if x.get('enrich_status')=='ambiguous']
print(f'\n姓名歧义需人工确认：{len(amb)} 位')
for n in amb: print('  -', n)

# 新状态：目标机构只在该作者历史记录里出现过，从未是最近一次隶属 —— 疑似已转校，
# 需人工核实（不会被当成 ok 自动采信，所以不会混进上面的「结果分布」的 ok 里）
moved_flag = [x['name'] for x in d if x.get('enrich_status') == 'possible_move']
print(f'\n🚚 疑似已转校（仅历史机构匹配，需人工核实）：{len(moved_flag)} 位')
for n in moved_flag: print('  -', n)

# 二次校验：即使标了 ok 的记录，也再对一次机构是否吻合（双重保险）
insts = lambda x: (x.get('effective_institutions') or x.get('last_known_institutions') or [])
moved = [(x['name'], x['institution'], insts(x))
         for x in d if x.get('enrich_status')=='ok' and insts(x)
         and x['institution'] and not any(x['institution'].split()[0].lower() in str(i).lower()
                                          for i in insts(x))]
print(f'\n⚠️ 标记为 ok 但机构仍对不上（少见，建议抽查）：{len(moved)} 位')
for n, old, new in moved[:20]: print(f'  - {n}：表格记「{old}」，OpenAlex 显示「{new}」')


---
## 第 8 步：生成新的 Excel（不会改动你原来的文件）

会新建一个文件，把 OpenAlex 的结果作为**新增列**追加在最右边，
原有的所有列和你的人工标记都不会被覆盖。

---
## 第 8 步：可信度评估（关键）

自动匹配的错误是**静默**的——匹配错的记录看起来和匹配对的一模一样。
这一步给每条结果打分并说明理由，让你知道**哪些可以直接用、哪些必须人工确认**。

评分依据包括：匹配方式、机构是否一致、合并了几条记录、论文期刊是否跨领域、
研究主题是否对口、任教起始年是否合理。

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, "confidence.py", "enriched.json",
                    "--out", "enriched_scored.json", "--field", "finance"],
                   capture_output=True, text=True)
print(p.stdout[-6000:])
print(p.stderr[-2000:])

---
## 第 8 步：可信度评估

对每条结果打分并给出理由，让你知道**哪些可以直接用、哪些必须人工确认**。
评分依据：匹配方式、机构是否吻合、合并记录数、论文是否跨领域、是否有近三年论文、是否有 ORCID。

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, "confidence.py", "enriched.json", "--out", "enriched.json"],
                   capture_output=True, text=True)
print(p.stdout[-6000:]); print(p.stderr[-2000:])

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, "merge_to_excel.py", XLSX, "enriched_scored.json",
                    "--sheet", SHEET, "--mailto", MAILTO, "--out", "结果_OA补全.xlsx"],
                   capture_output=True, text=True)
print(p.stdout); print(p.stderr)


---
## 第 9 步：生成人工核对清单

对于没能自动确认身份的教授，这一步会生成**可直接点开的 OpenAlex 链接**，
每个链接已经带好「姓名 + 机构」双重筛选，打开就能看到该机构内的同名作者，几秒钟即可确认。

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, "openalex_links.py", "--roster", "enriched.json",
                    "--mailto", MAILTO],
                   capture_output=True, text=True)
open("待人工确认.md", "w", encoding="utf-8").write(p.stdout)
print(p.stdout[:4000])
print("\n（完整内容已存为 待人工确认.md；默认列出所有非 ok 状态的行，"
      "不再只筛 ambiguous，possible_move / needs_review_contaminated 也会包含在内）")


In [ ]:
from google.colab import files
files.download("待人工确认.md")

In [ ]:
from google.colab import files
files.download('结果_OA补全.xlsx')
files.download('enriched.json')

In [ ]:
# 如需单独下载原始 json，运行上一格即可
print('完成')

---
## 下一步

把下载到的 `enriched.json` 发回给 Claude，说一句：

> 这是 batch_enrich 跑出来的结果，帮我把近三年发表、任教起始年、合著关系填回表格，并重新打分排名。

**歧义名单**（上面打印出来的那些）如果不多，也可以一起贴给 Claude 人工确认。

---
## 附：疑难人名的反向查找

对 Yingying Li、Wei Wang 这类常见姓名，「按姓名搜索再筛机构」不可靠。
反过来做：**先列出该机构在该领域发过论文的所有作者，再从中找名字**。

改下面的 `INST` 和 `NAME` 即可。

In [ ]:
import subprocess, sys
INST  = "HKUST"          # 机构（缩写即可）
FIELD_DEMO = FIELD        # 默认沿用上面配置区选好的领域；如需单独换一个领域测试，改这行即可
NAME  = "Yingying Li"    # 要找的人；留空则列出该机构该领域的全部作者

cmd = [sys.executable, "reverse_lookup.py", "--institution", INST,
       "--field", FIELD_DEMO, "--mailto", MAILTO, "--years", "6"]
if NAME:
    cmd += ["--name", NAME]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-6000:]); print(p.stderr[-2000:])
